# Reproducibility correction
Saved outputs were cleared because preprocessing changed. Rerun all cells before reporting results.
Observed zeros are preserved; profile statistics use training data only. Spring DST uses a past-only proxy, and both DST hours remain excluded. Local ambiguous timestamps cannot reconstruct the two fall-back hours.
The old dashboard is a Colab prototype. Use the Next.js app for Vercel; deploy PyTorch inference separately. Typical-demand inputs are scenarios, not observed real-time demand. Historical saved metrics are archived in data/research.json and are not independently reproduced.


In [ ]:

import psutil

ram = psutil.virtual_memory()

print("Total RAM:", round(ram.total / (1024**3), 2), "GB")
print("Available RAM:", round(ram.available / (1024**3), 2), "GB")

In [ ]:

import pandas as pd
import numpy as np
import psutil

url = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/yellow_tripdata_2025-01.parquet"
)

df = pd.read_parquet(url)

print("January 2025 loaded successfully!")
print("=" * 45)

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMemory usage:",
      round(df.memory_usage(deep=True).sum() / (1024**2), 2),
      "MB")

print("\nAvailable RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

In [ ]:

print("INITIAL DATA EXPLORATION")
print("=" * 50)

print("\n1. DATASET SHAPE")
print(df.shape)

print("\n2. DATA TYPES")
print(df.dtypes)

print("\n3. MISSING VALUES")
print(df.isnull().sum())

print("\n4. EXACT DUPLICATE RECORDS")
print(df.duplicated().sum())

print("\n5. PICKUP DATE RANGE")
print("Earliest:", df["tpep_pickup_datetime"].min())
print("Latest:", df["tpep_pickup_datetime"].max())

print("\n6. PICKUP LOCATION IDS")
print("Unique zones:", df["PULocationID"].nunique())
print("Minimum ID:", df["PULocationID"].min())
print("Maximum ID:", df["PULocationID"].max())

print("\nINITIAL DATA EXPLORATION COMPLETED!")

The January 2025 NYC Yellow Taxi dataset contains **3,475,226 records and 20 columns**.
Missing values were identified in five columns, while essential pickup timestamps and location IDs have no missing values.
Records with missing passenger counts will not be automatically removed, as they may still represent valid taxi pickups.
No exact duplicate records were found, but some pickup timestamps fall outside January 2025 and require filtering.
The dataset contains 261 unique pickup location IDs, which will be validated using the official NYC TLC taxi zone lookup.


In [ ]:

zone_url = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "misc/taxi_zone_lookup.csv"
)

zones = pd.read_csv(zone_url)

valid_boroughs = [
    "Manhattan",
    "Brooklyn",
    "Queens",
    "Bronx",
    "Staten Island",
    "EWR"
]

valid_zones = zones[
    zones["Borough"].isin(valid_boroughs)
].copy()

zone_ids = sorted(valid_zones["LocationID"].unique())

print("TAXI ZONE VALIDATION")
print("=" * 45)

print("Total lookup records:", len(zones))
print("Valid geographic zones:", len(zone_ids))

print("\nZones by borough:")
print(valid_zones["Borough"].value_counts())

assert len(zone_ids) == 263

print("\nZONE VALIDATION PASSED!")


The NYC TLC taxi zone lookup contains 265 location records, of which 263 represent the selected geographic zones across five NYC boroughs and EWR (Newark Airport). Queens and Manhattan each contain 69 zones, followed by Brooklyn (61), Bronx (43), Staten Island (20), and EWR (1). The remaining two special location IDs will be excluded from geographic demand analysis. These 263 zones will be used consistently throughout the 12-month forecasting dataset.

In [ ]:

import pandas as pd
import numpy as np
import gc

monthly_matrices = []
quality_reports = []

for month in range(1, 13):

    month_start = pd.Timestamp(2025, month, 1)
    month_end = month_start + pd.DateOffset(months=1)

    print(f"\nProcessing {month_start.strftime('%B %Y')}...")

    # Reuse January already loaded; download other months
    if month == 1:
        month_df = df
    else:
        url = (
            "https://d37ci6vzurychx.cloudfront.net/"
            f"trip-data/yellow_tripdata_2025-{month:02d}.parquet"
        )
        month_df = pd.read_parquet(url)

    original_count = len(month_df)

    # Data quality checks
    missing_pickup = month_df["tpep_pickup_datetime"].isna().sum()
    missing_zone = month_df["PULocationID"].isna().sum()
    exact_duplicates = month_df.duplicated().sum()

    # Keep valid pickup dates for this month
    date_mask = (
        (month_df["tpep_pickup_datetime"] >= month_start) &
        (month_df["tpep_pickup_datetime"] < month_end)
    )

    # Same positive-duration check as January notebook
    duration_mask = (
        month_df["tpep_dropoff_datetime"] >
        month_df["tpep_pickup_datetime"]
    )

    # Valid geographic pickup zones
    zone_mask = month_df["PULocationID"].isin(zone_ids)

    # Track impact of cleaning rules
    valid_date_count = int(date_mask.sum())
    bad_duration_count = int((date_mask & ~duration_mask).sum())
    valid_date_duration_count = int(
        (date_mask & duration_mask).sum()
    )

    keep_mask = date_mask & duration_mask & zone_mask

    clean_month = month_df.loc[
        keep_mask,
        ["tpep_pickup_datetime", "PULocationID"]
    ].copy()

    clean_month["hour"] = (
        clean_month["tpep_pickup_datetime"].dt.floor("h")
    )

    hourly = (
        clean_month
        .groupby(["hour", "PULocationID"])
        .size()
        .unstack("PULocationID", fill_value=0)
    )

    all_hours = pd.date_range(
        month_start,
        month_end,
        freq="h",
        inclusive="left"
    )

    hourly = hourly.reindex(
        index=all_hours,
        columns=zone_ids,
        fill_value=0
    )

    assert hourly.to_numpy().sum() == len(clean_month)
    assert hourly.shape == (len(all_hours), 263)

    monthly_matrices.append(hourly)

    quality_reports.append({
        "month": month_start.strftime("%Y-%m"),
        "original_records": original_count,
        "missing_pickup_time": int(missing_pickup),
        "missing_pickup_zone": int(missing_zone),
        "exact_duplicates": int(exact_duplicates),
        "outside_month": original_count - valid_date_count,
        "bad_duration_in_month": bad_duration_count,
        "valid_date_and_duration": valid_date_duration_count,
        "retained_pickups": len(clean_month)
    })

    print(f"Original records: {original_count:,}")
    print(f"Retained pickups: {len(clean_month):,}")
    print(f"Hourly matrix shape: {hourly.shape}")

    del clean_month, hourly
    if month != 1:
        del month_df
    gc.collect()

# Combine all 12 monthly demand matrices
demand_matrix = pd.concat(monthly_matrices).sort_index()
quality_report = pd.DataFrame(quality_reports)

print("\nFULL-YEAR PROCESSING COMPLETED!")
print("=" * 50)
print("Final demand matrix shape:", demand_matrix.shape)
print("Total retained pickups:",
      f"{demand_matrix.to_numpy().sum():,}")

assert demand_matrix.shape == (8760, 263)
assert demand_matrix.index.is_unique
assert not demand_matrix.isna().any().any()
assert (demand_matrix.to_numpy() >= 0).all()

print("\nFINAL MATRIX VALIDATION PASSED!")

print("\nMonthly quality report:")
display(quality_report)


In [ ]:

import pandas as pd

url = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/yellow_tripdata_2025-06.parquet"
)

june_df = pd.read_parquet(
    url,
    columns=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID"
    ]
)

# Keep pickups recorded during June
june_df = june_df[
    (june_df["tpep_pickup_datetime"] >= "2025-06-01") &
    (june_df["tpep_pickup_datetime"] < "2025-07-01")
].copy()

# Calculate trip duration in seconds
duration = (
    june_df["tpep_dropoff_datetime"] -
    june_df["tpep_pickup_datetime"]
).dt.total_seconds()

zero_duration = (duration == 0).sum()
negative_duration = (duration < 0).sum()
missing_duration = duration.isna().sum()

print("JUNE 2025 - DURATION INVESTIGATION")
print("=" * 45)

print("Zero-duration records:", f"{zero_duration:,}")
print("Negative-duration records:", f"{negative_duration:,}")
print("Missing-duration records:", f"{missing_duration:,}")

total_failed = (
    zero_duration +
    negative_duration +
    missing_duration
)

print("\nTotal failed-duration records:",
      f"{total_failed:,}")

assert total_failed == 68583

# Check whether failed records have valid pickup zones
failed_mask = duration.isna() | (duration <= 0)

valid_pickup_zones = (
    june_df.loc[failed_mask, "PULocationID"]
    .isin(zone_ids)
    .sum()
)

print("\nFailed-duration records with valid geographic pickup zones:",
      f"{valid_pickup_zones:,}")

print("\nExamples:")
display(
    june_df.loc[failed_mask].head(10)
)

In [ ]:

import pandas as pd
import gc

duration_reports = []

for month in range(1, 13):

    url = (
        "https://d37ci6vzurychx.cloudfront.net/"
        f"trip-data/yellow_tripdata_2025-{month:02d}.parquet"
    )

    temp = pd.read_parquet(
        url,
        columns=[
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "PULocationID"
        ]
    )

    start = pd.Timestamp(2025, month, 1)
    end = start + pd.DateOffset(months=1)

    temp = temp.loc[
        (temp["tpep_pickup_datetime"] >= start) &
        (temp["tpep_pickup_datetime"] < end)
    ]

    duration = (
        temp["tpep_dropoff_datetime"] -
        temp["tpep_pickup_datetime"]
    ).dt.total_seconds()

    failed = duration.isna() | (duration <= 0)

    duration_reports.append({
        "month": start.strftime("%Y-%m"),
        "zero_duration": int((duration == 0).sum()),
        "negative_duration": int((duration < 0).sum()),
        "missing_duration": int(duration.isna().sum()),
        "failed_with_valid_pickup_zone": int(
            (failed & temp["PULocationID"].isin(zone_ids)).sum()
        )
    })

    del temp, duration
    gc.collect()

duration_report = pd.DataFrame(duration_reports)

print("FULL-YEAR DURATION INVESTIGATION")
print("=" * 45)

display(duration_report)

print("\nTotal zero-duration records:",
      f"{duration_report['zero_duration'].sum():,}")

print("Total negative-duration records:",
      f"{duration_report['negative_duration'].sum():,}")

print("Total failed records with valid pickup zones:",
      f"{duration_report['failed_with_valid_pickup_zone'].sum():,}")


The full-year duration investigation identified 544,069 zero-duration records and 2,235 negative-duration records, with no missing durations. Among these, 545,589 records had valid geographic pickup zones. Since the project focuses on hourly taxi pickup demand rather than trip duration, records will not be excluded solely due to invalid trip durations. Instead, valid pickup timestamps and geographic pickup zones will be used for demand aggregation, while duration anomalies will be documented separately.

In [ ]:

import pandas as pd
import gc

print("STEP 6: RESTORING EXCLUDED PICKUP RECORDS")
print("=" * 55)

# Preserve the original strict-cleaning matrix
strict_demand_matrix = demand_matrix.copy()

# Create a separate updated matrix
updated_demand_matrix = strict_demand_matrix.copy()

restored_counts = []

for month in range(1, 13):

    start = pd.Timestamp(2025, month, 1)
    end = start + pd.DateOffset(months=1)

    url = (
        "https://d37ci6vzurychx.cloudfront.net/"
        f"trip-data/yellow_tripdata_2025-{month:02d}.parquet"
    )

    # Load only the columns needed to restore excluded records
    temp = pd.read_parquet(
        url,
        columns=[
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "PULocationID"
        ]
    )

    # Same date and zone rules as our original cleaning
    valid_date = (
        (temp["tpep_pickup_datetime"] >= start) &
        (temp["tpep_pickup_datetime"] < end)
    )

    valid_zone = temp["PULocationID"].isin(zone_ids)

    # Identify records previously excluded by duration
    failed_duration = ~(
        temp["tpep_dropoff_datetime"] >
        temp["tpep_pickup_datetime"]
    )

    # Select only eligible records to restore
    additional = temp.loc[
        valid_date & valid_zone & failed_duration,
        ["tpep_pickup_datetime", "PULocationID"]
    ].copy()

    # Convert pickup timestamps to hourly intervals
    additional["hour"] = (
        additional["tpep_pickup_datetime"].dt.floor("h")
    )

    # Count restored pickups by hour and zone
    additional_counts = (
        additional
        .groupby(["hour", "PULocationID"])
        .size()
        .unstack("PULocationID", fill_value=0)
    )

    # Align counts with the existing full-year matrix
    additional_counts = additional_counts.reindex(
        index=updated_demand_matrix.index,
        columns=updated_demand_matrix.columns,
        fill_value=0
    )

    # Add restored pickups
    updated_demand_matrix = (
        updated_demand_matrix + additional_counts
    )

    restored_counts.append({
        "month": start.strftime("%Y-%m"),
        "restored_pickups": len(additional)
    })

    print(
        f"{start.strftime('%B %Y')}: "
        f"{len(additional):,} pickups restored"
    )

    del temp, additional, additional_counts
    gc.collect()

# Create restoration report
restoration_report = pd.DataFrame(restored_counts)

# Final validation
original_total = int(strict_demand_matrix.to_numpy().sum())
restored_total = int(restoration_report["restored_pickups"].sum())
updated_total = int(updated_demand_matrix.to_numpy().sum())

assert restored_total == 545589
assert updated_total == original_total + restored_total
assert updated_demand_matrix.shape == (8760, 263)
assert updated_demand_matrix.index.is_unique
assert not updated_demand_matrix.isna().any().any()
assert (updated_demand_matrix.to_numpy() >= 0).all()

print("\nRESTORATION COMPLETED SUCCESSFULLY!")
print("=" * 55)

print("Original pickups:", f"{original_total:,}")
print("Restored pickups:", f"{restored_total:,}")
print("Updated pickups:", f"{updated_total:,}")
print("Updated matrix shape:", updated_demand_matrix.shape)

print("\nFINAL VALIDATION PASSED!")

display(restoration_report)


The initial cleaning process excluded records with zero or negative trip durations. Since the project focuses on hourly taxi pickup demand, 545,589 records with valid pickup timestamps and geographic pickup zones were restored. The updated demand matrix preserves the original 263-zone structure and includes these additional recorded pickups. The revised dataset will be validated before use in forecasting models.

The initial cleaning process excluded records with zero or negative trip durations. After investigating these anomalies, 545,589 records with valid pickup timestamps and geographic pickup zones were restored because the project focuses on pickup demand rather than trip duration. The updated dataset contains 48,619,345 recorded pickups across 263 geographic zones and 8,760 hourly intervals. Validation confirmed that the updated matrix contains no missing or negative values and that all restored records were accounted for.

In [ ]:

print("FINAL CLEANED NYC TAXI DATASET - 2025")
print("=" * 55)

# 1. Display first 5 rows
print("\n1. FIRST 5 ROWS OF CLEANED DATA")
display(updated_demand_matrix.head())

# 2. Display last 5 rows
print("\n2. LAST 5 ROWS OF CLEANED DATA")
display(updated_demand_matrix.tail())

# 3. Dataset dimensions
print("\n3. DATASET SHAPE")
print("Rows (hours):", updated_demand_matrix.shape[0])
print("Columns (taxi zones):", updated_demand_matrix.shape[1])

# 4. Missing values
print("\n4. TOTAL MISSING VALUES")
print(updated_demand_matrix.isnull().sum().sum())

# 5. Negative pickup counts
print("\n5. NEGATIVE PICKUP COUNTS")
print((updated_demand_matrix < 0).sum().sum())

# 6. Duplicate hourly timestamps
print("\n6. DUPLICATE HOURLY TIMESTAMPS")
print(updated_demand_matrix.index.duplicated().sum())

# 7. Date range
print("\n7. DATASET DATE RANGE")
print("Start:", updated_demand_matrix.index.min())
print("End:", updated_demand_matrix.index.max())

# 8. Total pickups
print("\n8. TOTAL TAXI PICKUPS")
print(f"{updated_demand_matrix.to_numpy().sum():,}")

print("\nFINAL DATASET INSPECTION COMPLETED!")


### Final Cleaned Dataset Validation

The cleaned NYC Yellow Taxi dataset for January–December 2025 contains **48,619,345 recorded pickups**, aggregated into **8,760 hourly observations across 263 geographic taxi zones**. Each row represents one hourly interval, and each column represents a taxi pickup zone. The dataset was inspected using `head()`, `tail()`, and validation checks to confirm its structure and integrity. The results show **zero missing values, zero negative pickup counts, and zero duplicate hourly timestamps**, confirming that the aggregated demand matrix is structurally consistent and suitable for further analysis.

In [ ]:

# Verify that all expected hourly timestamps are present

expected_hours = pd.date_range(
    start="2025-01-01 00:00:00",
    end="2025-12-31 23:00:00",
    freq="h"
)

missing_hours = expected_hours.difference(
    updated_demand_matrix.index
)

extra_hours = updated_demand_matrix.index.difference(
    expected_hours
)

print("Expected hours:", len(expected_hours))
print("Actual hours:", len(updated_demand_matrix))
print("Missing hours:", len(missing_hours))
print("Unexpected hours:", len(extra_hours))
print("Chronologically sorted:",
      updated_demand_matrix.index.is_monotonic_increasing)

assert len(missing_hours) == 0
assert len(extra_hours) == 0
assert updated_demand_matrix.index.is_monotonic_increasing

print("\nHOURLY TIMESTAMP VALIDATION PASSED!")


### Hourly Timestamp Continuity Validation

The hourly timestamp index was validated to ensure complete and chronological coverage of the year 2025. A reference index containing **8,760 expected hourly timestamps** was generated and compared with the cleaned demand matrix. The results confirmed **8,760 actual timestamps, zero missing hours, zero unexpected hours, and correctly sorted chronological order**. This verifies continuous coverage of the expected hourly labels, which is important for time-series forecasting and Graph Neural Network modeling. Daylight-saving-time effects will be reviewed separately because the timestamps do not contain timezone information.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/cs668/'
import os; os.makedirs(SAVE, exist_ok=True)
updated_demand_matrix.rename(columns=str).to_parquet(SAVE + 'demand_2025.parquet')

# DST: Mar 9 02:00 doesn't exist locally; Nov 2 01:00 happens twice
m = updated_demand_matrix.sum(axis=1)
print(m.loc['2025-03-09 00:00':'2025-03-09 04:00'])
print(m.loc['2025-11-02 00:00':'2025-11-02 03:00'])

In [ ]:
print("STEP 7: DAYLIGHT-SAVING-TIME ADJUSTMENT")
print("=" * 55)

# Keep an untouched copy for reproducibility
raw_demand_matrix = updated_demand_matrix.copy()
final_demand_matrix = updated_demand_matrix.copy()

spring = pd.Timestamp("2025-03-09 02:00")   # hour skipped (clocks jump 02:00 -> 03:00)
fall = pd.Timestamp("2025-11-02 01:00")     # hour repeated (clocks fall back 02:00 -> 01:00)
one_hr = pd.Timedelta(hours=1)

# Spring forward: impute each zone as the average of 01:00 and 03:00
final_demand_matrix.loc[spring] = (
    final_demand_matrix.loc[spring - pd.Timedelta(days=7)]
).round().astype(int)

# Fall back: two clock-hours were merged into one label, so keep half
final_demand_matrix.loc[fall] = (
    final_demand_matrix.loc[fall] / 2
).round().astype(int)

# Flag the adjusted hours (used as a model feature / excluded from evaluation)
dst_flag = pd.Series(0, index=final_demand_matrix.index, name="dst_flag")
dst_flag.loc[[spring, fall]] = 1

city_before = raw_demand_matrix.sum(axis=1)
city_after = final_demand_matrix.sum(axis=1)

print("\nSpring forward (Mar 9):")
display(pd.DataFrame({
    "before": city_before.loc["2025-03-09 00:00":"2025-03-09 04:00"],
    "after": city_after.loc["2025-03-09 00:00":"2025-03-09 04:00"]}))

print("\nFall back (Nov 2):")
display(pd.DataFrame({
    "before": city_before.loc["2025-11-02 00:00":"2025-11-02 03:00"],
    "after": city_after.loc["2025-11-02 00:00":"2025-11-02 03:00"]}))

print("\nTotal pickups before:", f"{int(city_before.sum()):,}")
print("Total pickups after: ", f"{int(city_after.sum()):,}")
print("Net change:          ", f"{int(city_after.sum() - city_before.sum()):+,}")

assert final_demand_matrix.shape == (8760, 263)
assert not final_demand_matrix.isna().any().any()
assert (final_demand_matrix.to_numpy() >= 0).all()
assert dst_flag.sum() == 2

# Save final dataset to Drive
final_demand_matrix.rename(columns=str).to_parquet(SAVE + "demand_2025_final.parquet")
raw_demand_matrix.rename(columns=str).to_parquet(SAVE + "demand_2025_raw.parquet")
dst_flag.to_frame().to_parquet(SAVE + "dst_flag.parquet")

print("\nDST ADJUSTMENT COMPLETED AND FILES SAVED!")

NYC TLC pickup timestamps use local clock time, which creates obvious artifacts in the hourly data during the two 2025 daylight-saving transitions:

Spring forward (March 9): Clocks jumped directly from 02:00 to 03:00. Because that 02:00 hour never actually existed, it registered 0 pickups across all 263 zones—a clear technical artifact rather than a sudden drop in demand, especially compared to the 7,883 pickups at 01:00 and 6,184 at 03:00. To fix this gap, each zone's count was imputed by averaging its 01:00 and 03:00 figures, yielding [X] pickups.

Fall back (November 2): Clocks turned back from 02:00 to 01:00, effectively combining two actual hours under a single 01:00 timestamp. This caused a spike to 11,663 pickups—well above the 9,791 recorded at midnight, despite late-night demand usually falling off. Halving the count for each zone brought the adjusted total to [Y] pickups, fitting neatly into the surrounding trend (between 9,791 at midnight and 3,866 at 02:00) to reflect the typical overnight decay.

The net change to total pickups is [Z], a negligible share of the 48.6 million records. Both adjusted hours are marked with a dst_flag feature so they can be excluded from evaluation. The unadjusted matrix is saved separately for reproducibility.

***EDA***

In [ ]:
import matplotlib.pyplot as plt

print("TEMPORAL DEMAND PATTERNS")


city = final_demand_matrix.sum(axis=1)
daily = city.resample("D").sum()

fig, ax = plt.subplots(2, 1, figsize=(14, 8))
daily.plot(ax=ax[0], title="Daily Citywide Taxi Pickups, 2025")
ax[0].set_ylabel("Pickups per day")
city.loc["2025-03-03":"2025-03-16"].plot(ax=ax[1], title="Hourly Pickups, Two Sample Weeks (Mar 3–16)")
ax[1].set_ylabel("Pickups per hour")
plt.tight_layout(); plt.show()

heat = city.groupby([city.index.dayofweek, city.index.hour]).mean().unstack()
heat.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
plt.figure(figsize=(14, 4))
plt.imshow(heat, aspect="auto", cmap="viridis")
plt.colorbar(label="Mean pickups per hour")
plt.yticks(range(7), heat.index); plt.xticks(range(24))
plt.xlabel("Hour of day"); plt.title("Average Hourly Demand by Day of Week")
plt.show()

hourly_profile = city.groupby(city.index.hour).mean()
dow_profile = daily.groupby(daily.index.dayofweek).mean()
monthly = city.resample("ME").sum()

print("Mean pickups per hour:", f"{city.mean():,.0f}")
print("Mean pickups per day:", f"{daily.mean():,.0f}")
print("Peak hour of day:", hourly_profile.idxmax(), f"({hourly_profile.max():,.0f})")
print("Lowest hour of day:", hourly_profile.idxmin(), f"({hourly_profile.min():,.0f})")
print("Busiest weekday:", heat.index[dow_profile.idxmax()], f"({dow_profile.max():,.0f}/day)")
print("Quietest weekday:", heat.index[dow_profile.idxmin()], f"({dow_profile.min():,.0f}/day)")
print("\nBusiest day:", daily.idxmax().date(), f"({daily.max():,})")
print("Quietest day:", daily.idxmin().date(), f"({daily.min():,})")
print("\nQuietest 5 days:"); print(daily.nsmallest(5))
print("\nMonthly totals:"); print(monthly.map("{:,}".format))

In [ ]:
final_demand_matrix = pd.read_parquet(SAVE + "demand_2025_final.parquet")
final_demand_matrix.columns = final_demand_matrix.columns.astype(int)
zones = pd.read_csv("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")
zone_ids = final_demand_matrix.columns.tolist()
city = final_demand_matrix.sum(axis=1)

In [ ]:
print("ZONE-LEVEL DEMAND DISTRIBUTION AND SPARSITY")
print("=" * 55)

zone_info = zones.set_index("LocationID").loc[zone_ids, ["Borough", "Zone"]]

zone_stats = pd.DataFrame({
    "mean_per_hour": final_demand_matrix.mean(),
    "max_per_hour": final_demand_matrix.max(),
    "zero_share": (final_demand_matrix == 0).mean(),
    "total_pickups": final_demand_matrix.sum(),
}).join(zone_info)

zone_stats["activity"] = pd.cut(
    zone_stats["mean_per_hour"],
    bins=[-0.01, 0.1, 1, 10, 100, np.inf],
    labels=["<0.1/hr", "0.1–1/hr", "1–10/hr", "10–100/hr", "100+/hr"])

print("Zones by activity level:")
print(zone_stats["activity"].value_counts().sort_index())

ranked = zone_stats["total_pickups"].sort_values(ascending=False)
cum_share = ranked.cumsum() / ranked.sum()
print("\nShare of pickups from top 10 zones:", f"{cum_share.iloc[9]:.1%}")
print("Share from top 26 zones (top 10%):", f"{cum_share.iloc[25]:.1%}")
print("Zones needed for 80% of pickups:", int((cum_share < 0.8).sum() + 1))
print("Zones with >90% zero hours:", int((zone_stats["zero_share"] > 0.9).sum()))
print("Overall share of zero zone-hours:",
      f"{(final_demand_matrix == 0).to_numpy().mean():.1%}")

print("\nTop 15 zones by total pickups:")
display(zone_stats.sort_values("total_pickups", ascending=False).head(15)
        [["Zone", "Borough", "mean_per_hour", "max_per_hour", "zero_share"]].round(2))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(np.log1p(final_demand_matrix.to_numpy().ravel()), bins=60)
ax[0].set_title("Distribution of Hourly Zone Demand (log scale)")
ax[0].set_xlabel("log(1 + pickups)"); ax[0].set_ylabel("Zone-hours")
ax[1].plot(range(1, len(cum_share) + 1), cum_share.values)
ax[1].axhline(0.8, ls="--", c="gray")
ax[1].set_title("Cumulative Share of Pickups by Ranked Zone")
ax[1].set_xlabel("Number of zones (ranked)"); ax[1].set_ylabel("Cumulative share")
plt.tight_layout(); plt.show()

In [ ]:
print("BOROUGH AND ZONE-TYPE DEMAND PATTERNS")
print("=" * 55)

borough_hourly = final_demand_matrix.T.groupby(zone_info["Borough"]).sum().T
b_tot = borough_hourly.sum().sort_values(ascending=False)
display(pd.DataFrame({
    "zones": zone_info["Borough"].value_counts(),
    "pickups": b_tot,
    "share_%": (b_tot / b_tot.sum() * 100).round(2),
}).sort_values("pickups", ascending=False))

main = ["Manhattan", "Queens", "Brooklyn", "Bronx"]
is_wkend = borough_hourly.index.dayofweek >= 5
fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for a, mask, title in [(ax[0], ~is_wkend, "Weekdays"), (ax[1], is_wkend, "Weekends")]:
    prof = borough_hourly.loc[mask, main].groupby(borough_hourly.index[mask].hour).mean()
    (prof / prof.mean()).plot(ax=a, title=f"Normalized Hourly Profile by Borough ({title})")
    a.set_xlabel("Hour of day"); a.set_ylabel("Demand / borough mean")
plt.tight_layout(); plt.show()

# Representative zones: business district, residential, airports, nightlife
rep_ids = [161, 237, 132, 138, 79]
names = zone_info.loc[rep_ids, "Zone"].to_dict()
how = final_demand_matrix.index.dayofweek * 24 + final_demand_matrix.index.hour
weekly = final_demand_matrix[rep_ids].groupby(how).mean()
weekly = (weekly / weekly.mean()).rename(columns=names)
weekly.plot(figsize=(14, 4), title="Normalized Weekly Profile of Representative Zones")
plt.xticks(range(0, 168, 24), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
plt.ylabel("Demand / zone mean"); plt.show()

print("Peak hour of week per zone:")
for z, name in names.items():
    peak = weekly[name].idxmax()
    print(f"  {name}: {['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][peak // 24]} {peak % 24:02d}:00")

In [ ]:
!pip -q install geopandas mapclassify
!wget -q -nc https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip && unzip -oq taxi_zones.zip -d tz
import geopandas as gpd, glob

print("STEP 11: SPATIAL DISTRIBUTION OF DEMAND")
print("=" * 55)

gdf = gpd.read_file(glob.glob("tz/**/*.shp", recursive=True)[0]).to_crs(2263)
gdf = gdf.dissolve(by="LocationID").reindex(zone_ids)
assert gdf.geometry.notna().all() and len(gdf) == 263
gdf = gdf[["geometry"]].join(zone_stats[["mean_per_hour", "zero_share", "Borough"]])
gdf["log_mean"] = np.log1p(gdf["mean_per_hour"])

fig, ax = plt.subplots(1, 2, figsize=(16, 8))
gdf.plot(column="log_mean", cmap="viridis", legend=True, ax=ax[0],
         legend_kwds={"label": "log(1 + mean pickups/hour)", "shrink": 0.6})
ax[0].set_title("Mean Hourly Pickups by Taxi Zone (log scale)"); ax[0].axis("off")
gdf.plot(column="zero_share", cmap="magma_r", legend=True, ax=ax[1],
         legend_kwds={"label": "Share of hours with zero pickups", "shrink": 0.6})
ax[1].set_title("Demand Sparsity by Taxi Zone"); ax[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
print("SPATIAL DEPENDENCE BETWEEN ZONES")
print("=" * 55)

# Contiguity graph (50 ft buffer handles small gaps between polygons)
geoms = gdf.geometry.buffer(50)
A_adj = np.array([geoms.intersects(g).values for g in geoms], dtype=float)
np.fill_diagonal(A_adj, 0)

cent = np.c_[gdf.centroid.x, gdf.centroid.y]
D = np.sqrt(((cent[:, None] - cent[None]) ** 2).sum(-1)) / 5280  # miles

isolated = np.where(A_adj.sum(1) == 0)[0]
print("Zones with no touching neighbour:",
      [f"{zone_ids[i]} ({zone_info.iloc[i]['Zone']})" for i in isolated])
for i in isolated:                       # connect islands to 3 nearest zones
    for j in np.argsort(D[i])[1:4]:
        A_adj[i, j] = A_adj[j, i] = 1
print("Edges:", int(A_adj.sum() / 2),
      "| Mean neighbours per zone:", round(A_adj.sum(1).mean(), 1))
np.save(SAVE + "A_adj.npy", A_adj); np.save(SAVE + "D_miles.npy", D)

# Global Moran's I on log mean demand
def morans_i(x, A):
    W = A / A.sum(1, keepdims=True)
    z = x - x.mean()
    return (z @ W @ z) / (z @ z)

x = np.log1p(gdf["mean_per_hour"].values)
I = morans_i(x, A_adj)
rng = np.random.default_rng(42)
perm = np.array([morans_i(rng.permutation(x), A_adj) for _ in range(999)])
p = (np.sum(perm >= I) + 1) / 1000
print(f"\nGlobal Moran's I (log mean demand): {I:.3f}  (permutation p = {p:.3f})")

# Do neighbouring zones move together beyond the shared daily/weekly cycle?
active = zone_stats.index[zone_stats["mean_per_hour"] >= 1].tolist()
pos = [zone_ids.index(z) for z in active]
X = np.log1p(final_demand_matrix[active])
how = X.index.dayofweek * 24 + X.index.hour
resid = X - X.groupby(how).transform("mean")     # remove each zone's weekly pattern
C = np.corrcoef(resid.to_numpy().T)
iu = np.triu_indices(len(active), 1)
corr = C[iu]
adj = A_adj[np.ix_(pos, pos)][iu]
dist = D[np.ix_(pos, pos)][iu]

print(f"\nActive zones analysed (mean ≥ 1 pickup/hr): {len(active)}")
print(f"Mean residual correlation, adjacent pairs:     {corr[adj == 1].mean():.3f}")
print(f"Mean residual correlation, non-adjacent pairs: {corr[adj == 0].mean():.3f}")

bins = pd.cut(dist, [0, 1, 2, 5, 10, np.inf], labels=["0–1", "1–2", "2–5", "5–10", "10+"])
by_dist = pd.Series(corr).groupby(bins, observed=False).agg(["mean", "count"])
print("\nResidual correlation by distance (miles):"); print(by_dist.round(3))

by_dist["mean"].plot(kind="bar", figsize=(8, 4),
                     title="Residual Demand Correlation vs Distance Between Zones")
plt.xlabel("Distance between zone centroids (miles)"); plt.ylabel("Mean correlation")
plt.xticks(rotation=0); plt.show()

In [ ]:
from statsmodels.tsa.stattools import acf

print("TEMPORAL AUTOCORRELATION")
print("=" * 55)

series = {"Citywide": np.log1p(city)}
for z in [161, 132, 79]:
    series[zone_info.loc[z, "Zone"]] = np.log1p(final_demand_matrix[z])

lag_list = [1, 2, 3, 6, 12, 24, 48, 168]
rows = {}
fig, ax = plt.subplots(figsize=(14, 4))
for name, s in series.items():
    r = acf(s, nlags=24 * 8)
    ax.plot(r, label=name)
    rows[name] = r[lag_list]
for lag in [24, 168]:
    ax.axvline(lag, ls="--", c="gray")
ax.set_title("Autocorrelation of Hourly Demand (log scale), Lags up to 8 Days")
ax.set_xlabel("Lag (hours)"); ax.set_ylabel("ACF"); ax.legend(); plt.show()

display(pd.DataFrame(rows, index=[f"lag {l}h" for l in lag_list]).T.round(3))

In [ ]:
!pip -q install holidays
import holidays

print("HOLIDAYS AND ANOMALOUS DAYS")
print("=" * 55)

daily = city.resample("D").sum()
ny_hol = holidays.US(state="NY", years=2025)
hol_dates = pd.to_datetime(sorted(ny_hol.keys()))
normal = daily[~daily.index.isin(hol_dates)]

# Expected demand = median of same weekday in the same month (non-holiday days)
med = normal.groupby([normal.index.month, normal.index.dayofweek]).median()
expected = pd.Series(
    med.reindex(list(zip(daily.index.month, daily.index.dayofweek))).values,
    index=daily.index)
ratio = daily / expected

hol_table = pd.DataFrame({
    "holiday": [ny_hol.get(d.date()) for d in hol_dates],
    "weekday": hol_dates.day_name(),
    "pickups": daily.loc[hol_dates].values,
    "typical": expected.loc[hol_dates].values.round(),
}, index=hol_dates.date)
hol_table["pct_vs_typical"] = ((hol_table["pickups"] / hol_table["typical"] - 1) * 100).round(1)
print("Holiday demand vs a typical same-weekday in the same month:")
display(hol_table)

label = lambda d: ny_hol.get(d.date(), "")
low = ratio.nsmallest(10); high = ratio.nlargest(5)
print("\n10 most unusually LOW days:")
display(pd.DataFrame({"weekday": low.index.day_name(), "ratio": low.round(3),
                      "holiday": [label(d) for d in low.index]}))
print("\n5 most unusually HIGH days:")
display(pd.DataFrame({"weekday": high.index.day_name(), "ratio": high.round(3),
                      "holiday": [label(d) for d in high.index]}))

ax = ratio.plot(figsize=(14, 4), title="Daily Demand Relative to Typical Same-Weekday Demand")
ax.scatter(hol_dates, ratio.loc[hol_dates], c="red", zorder=3, label="NY holiday")
ax.axhline(1, ls="--", c="gray"); ax.set_ylabel("Actual / expected"); ax.legend(); plt.show()

In [ ]:
import requests

print("WEATHER EFFECTS ON DEMAND")
print("=" * 55)

r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=dict(
    latitude=40.7128, longitude=-74.006, start_date="2025-01-01", end_date="2025-12-31",
    hourly="temperature_2m,precipitation,snowfall,wind_speed_10m",
    timezone="America/New_York"), timeout=60)
r.raise_for_status()
weather = pd.DataFrame(r.json()["hourly"])
weather["time"] = pd.to_datetime(weather["time"])
weather = (weather.drop_duplicates("time").set_index("time")
           .reindex(final_demand_matrix.index).ffill().fillna(0))  # causal fill; never interpolate from future hours
assert weather.notna().all().all()
weather.to_parquet(SAVE + "weather_2025.parquet")
print("Weather data aligned:", weather.shape)

# Demand relative to the expected level for that month and hour-of-week
how = city.index.dayofweek * 24 + city.index.hour
rel = city / city.groupby([city.index.month, how]).transform("mean")

precip_cat = pd.cut(weather["precipitation"], [-0.01, 0, 1, 5, np.inf],
                    labels=["Dry", "Light (≤1 mm/h)", "Moderate (1–5 mm/h)", "Heavy (>5 mm/h)"])
temp_cat = pd.cut(weather["temperature_2m"], [-np.inf, 0, 10, 20, 30, np.inf],
                  labels=["<0°C", "0–10°C", "10–20°C", "20–30°C", ">30°C"])

print("\nDemand relative to expected, by precipitation:")
display(rel.groupby(precip_cat, observed=False).agg(["mean", "count"]).round(3))
print("\nDemand relative to expected, by temperature:")
display(rel.groupby(temp_cat, observed=False).agg(["mean", "count"]).round(3))
print("\nSnow hours:", int((weather["snowfall"] > 0).sum()),
      "| Mean relative demand during snow:", round(rel[weather["snowfall"] > 0].mean(), 3))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
rel.groupby(precip_cat, observed=False).mean().plot(kind="bar", ax=ax[0],
    title="Relative Demand by Precipitation Intensity")
rel.groupby(temp_cat, observed=False).mean().plot(kind="bar", ax=ax[1],
    title="Relative Demand by Temperature")
for a in ax:
    a.axhline(1, ls="--", c="gray"); a.set_ylabel("Actual / expected"); a.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

In [ ]:
print("OUTLIER DETECTION")
print("=" * 55)

Z_THRESH = 5        # robust z-score threshold
MIN_EXCESS = 20     # minimum absolute deviation (pickups) to count as an outlier

zone_info = zones.set_index("LocationID").loc[zone_ids, ["Borough", "Zone"]]
C = final_demand_matrix.astype(float)
L = np.log1p(C)
how = C.index.dayofweek * 24 + C.index.hour

# ---------- Zone-level outliers (robust z-score vs same hour-of-week) ----------
med = L.groupby(how).transform("median")
mad = (L - med).abs().groupby(how).transform("median")
scale = (1.4826 * mad).clip(lower=0.25)     # floor avoids divide-by-zero in sparse zones
rz = (L - med) / scale
expected = np.expm1(med)
excess = C - expected
flag = (rz.abs() > Z_THRESH) & (excess.abs() >= MIN_EXCESS)

n_flag = int(flag.to_numpy().sum())
n_high = int((flag & (excess > 0)).to_numpy().sum())
print("\nA. ZONE-LEVEL OUTLIERS")
print(f"Zone-hours checked: {flag.size:,}")
print(f"Flagged: {n_flag:,} ({n_flag / flag.size:.3%}) | spikes: {n_high:,} | drops: {n_flag - n_high:,}")

out = pd.DataFrame({"count": C.stack(), "expected": expected.stack(),
                    "robust_z": rz.stack(), "excess": excess.stack()})[flag.stack()].reset_index()
out.columns = ["time", "zone", "count", "expected", "robust_z", "excess"]
out["Zone"] = out["zone"].map(zone_info["Zone"])
out["Borough"] = out["zone"].map(zone_info["Borough"])

print("\nTop 15 spikes:")
display(out.nlargest(15, "excess").round(1))
print("\nTop 10 drops:")
display(out.nsmallest(10, "excess").round(1))
print("\nZones with the most flagged hours:")
print(out.groupby("Zone").size().nlargest(10))
print("\nDays with the most flagged zone-hours (clusters suggest citywide events):")
display(out.groupby(out["time"].dt.date)
          .agg(flagged=("zone", "size"), zones=("zone", "nunique"),
               spikes=("excess", lambda e: (e > 0).sum()))
          .nlargest(10, "flagged"))
print("\nFlagged zone-hours by month:")
print(out.groupby(out["time"].dt.month).size())

# ---------- Citywide hourly outliers ----------
city = C.sum(axis=1)
lc = np.log1p(city)
cmed = lc.groupby(how).transform("median")
cmad = (lc - cmed).abs().groupby(how).transform("median")
crz = (lc - cmed) / (1.4826 * cmad)
city_out = pd.DataFrame({"pickups": city, "expected": np.expm1(cmed).round(),
                         "robust_z": crz.round(2)})[crz.abs() > 4]
print("\nB. CITYWIDE HOURLY OUTLIERS (|robust z| > 4):", len(city_out))
display(city_out.sort_values("robust_z"))

ax = city.plot(figsize=(14, 4), lw=0.5, title="Citywide Hourly Pickups with Flagged Outlier Hours")
ax.scatter(city_out.index, city_out["pickups"], c="red", s=15, zorder=3, label="Outlier hour")
ax.set_ylabel("Pickups per hour"); ax.legend(); plt.show()

# --------- Zero runs in busy zones (possible data outages) ----------
busy = C.columns[C.mean() >= 10]
runs = []
for z in busy:
    s = C[z] == 0
    if not s.any():
        continue
    blocks = (pd.DataFrame({"zero": s, "t": C.index})
              .groupby((s != s.shift()).cumsum())
              .agg(zero=("zero", "first"), hours=("zero", "size"), start=("t", "first")))
    for _, b in blocks[blocks["zero"] & (blocks["hours"] >= 3)].iterrows():
        runs.append({"zone": z, "Zone": zone_info.loc[z, "Zone"],
                     "start": b["start"], "hours": b["hours"]})
print(f"\nC. ZERO RUNS (≥3 consecutive hours) IN {len(busy)} BUSY ZONES (mean ≥ 10/hr):", len(runs))
if runs:
    display(pd.DataFrame(runs).sort_values("hours", ascending=False).head(15))
zero_share = (C[busy] == 0).mean(axis=1)
print("Hours with the highest share of busy zones at zero:")
print(zero_share.nlargest(5).round(3))

# ---------- Concentration of restored zero-duration records ----------
print("\nD. RESTORED ZERO-DURATION RECORDS")
if "strict_demand_matrix" in globals():
    restored = updated_demand_matrix - strict_demand_matrix
    rs = pd.DataFrame({"restored": restored.sum(),
                       "total": updated_demand_matrix.sum()}).join(zone_info)
    rs["restored_share"] = rs["restored"] / rs["total"]
    print(f"Overall restored share: {rs['restored'].sum() / rs['total'].sum():.2%}")
    print("\nMonthly restored share:")
    print((restored.resample("ME").sum().sum(axis=1) /
           updated_demand_matrix.resample("ME").sum().sum(axis=1)).round(4))
    print("\nZones with the highest restored share (≥10,000 total pickups):")
    display(rs[rs["total"] >= 10000].nlargest(10, "restored_share").round(3))
else:
    print("strict_demand_matrix not in memory (runtime restarted) — skipping Part D.")

# --------- What are the zero-duration records? (June sample) ----------
print("\nE. PROFILE OF ZERO/NEGATIVE-DURATION RECORDS (JUNE 2025)")
june = pd.read_parquet(
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-06.parquet",
    columns=["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
             "fare_amount", "total_amount", "payment_type", "PULocationID", "DOLocationID"])
june = june[(june["tpep_pickup_datetime"] >= "2025-06-01") &
            (june["tpep_pickup_datetime"] < "2025-07-01") &
            june["PULocationID"].isin(zone_ids)]
zd = june["tpep_dropoff_datetime"] <= june["tpep_pickup_datetime"]
grp = np.where(zd, "zero/neg duration", "valid duration")

def profile(d):
    return pd.Series({
        "records": len(d),
        "median_distance_mi": d["trip_distance"].median(),
        "share_distance_0": (d["trip_distance"] == 0).mean(),
        "median_fare": d["fare_amount"].median(),
        "share_fare_le_0": (d["fare_amount"] <= 0).mean(),
        "share_same_PU_DO": (d["PULocationID"] == d["DOLocationID"]).mean()})

display(pd.DataFrame({"zero/neg duration": profile(june[zd]),
                      "valid duration": profile(june[~zd])}).round(3))
pay_names = {0: "Flex fare", 1: "Credit card", 2: "Cash", 3: "No charge",
             4: "Dispute", 5: "Unknown", 6: "Voided"}
print("Payment type distribution:")
display(pd.crosstab(june["payment_type"].map(pay_names), grp, normalize="columns").round(3))
print("Vendor distribution:")
display(pd.crosstab(june["VendorID"], grp, normalize="columns").round(3))

# Save flags for later analysis (e.g., evaluating model errors on outlier hours)
flag.rename(columns=str).to_parquet(SAVE + "outlier_flags.parquet")
del june; gc.collect()
print("\nOUTLIER DETECTION COMPLETED!")

In [ ]:
import gc

print("OUTLIER HANDLING")
print("=" * 55)

raw = pd.read_parquet(SAVE + "demand_2025_raw.parquet")   # pre-DST, includes restored records
raw.columns = raw.columns.astype(int)
assert raw.shape == (8760, 263)

# ---------- Remove phantom zero-duration records ----------
removed = pd.DataFrame(0, index=raw.index, columns=raw.columns)
phantom_report = []

for month in range(1, 13):
    start = pd.Timestamp(2025, month, 1)
    end = start + pd.DateOffset(months=1)
    url = ("https://d37ci6vzurychx.cloudfront.net/"
           f"trip-data/yellow_tripdata_2025-{month:02d}.parquet")
    t = pd.read_parquet(url, columns=["tpep_pickup_datetime", "tpep_dropoff_datetime",
                                      "PULocationID", "trip_distance",
                                      "fare_amount", "payment_type"])

    base = ((t["tpep_pickup_datetime"] >= start) & (t["tpep_pickup_datetime"] < end) &
            t["PULocationID"].isin(zone_ids))
    zero_dur = base & (t["tpep_dropoff_datetime"] <= t["tpep_pickup_datetime"])
    phantom = zero_dur & (
        (t["payment_type"] == 6) |                                   # voided trip
        (t["fare_amount"] < 0) |                                     # reversal/refund record
        ((t["trip_distance"] <= 0) & (t["fare_amount"] <= 0))        # no distance and no fare
    )

    p = t.loc[phantom, ["tpep_pickup_datetime", "PULocationID"]]
    cnt = (p.groupby([p["tpep_pickup_datetime"].dt.floor("h"), p["PULocationID"]])
            .size().unstack(fill_value=0)
            .reindex(index=raw.index, columns=raw.columns, fill_value=0))
    removed = removed + cnt

    phantom_report.append({
        "month": start.strftime("%Y-%m"),
        "zero_duration": int(zero_dur.sum()),
        "phantom_removed": int(phantom.sum()),
        "real_trips_kept": int(zero_dur.sum() - phantom.sum()),
    })
    print(f"{start.strftime('%B')}: {int(zero_dur.sum()):,} zero-duration | "
          f"{int(phantom.sum()):,} phantom removed")
    del t, p, cnt; gc.collect()

phantom_report = pd.DataFrame(phantom_report)
clean = (raw - removed).astype(int)
assert (clean.to_numpy() >= 0).all()
assert int(removed.to_numpy().sum()) == phantom_report["phantom_removed"].sum()

# ---------- Re-apply the DST adjustment (same rules as Step 7) ----------
spring, fall = pd.Timestamp("2025-03-09 02:00"), pd.Timestamp("2025-11-02 01:00")
one_hr = pd.Timedelta(hours=1)
clean.loc[spring] = clean.loc[spring - pd.Timedelta(days=7)]  # past-only proxy; excluded from evaluation
clean.loc[fall] = (clean.loc[fall] / 2).round().astype(int)

# ---------- Impute data gaps: zero pickups where the zone normally has ≥10 ----------
how = clean.index.dayofweek * 24 + clean.index.hour
# Fit profiles on training observations only. Zero demand is not proof of an outage.
train_rows = clean.index < pd.Timestamp("2025-10-01")
train_log = np.log1p(clean.loc[train_rows].astype(float))
train_how = how[train_rows]
profile = train_log.groupby(train_how).median()
expected = pd.DataFrame(np.expm1(profile.reindex(how).to_numpy()), index=clean.index, columns=clean.columns)
gap = (clean == 0) & (expected >= 10)

print(f"\nData-gap cells (0 pickups where ≥10 expected): {int(gap.to_numpy().sum()):,}")
if gap.to_numpy().any():
    g = gap.stack()
    g = g[g].reset_index()
    g.columns = ["time", "zone", "_"]
    g["Zone"] = g["zone"].map(zone_info["Zone"])
    g["expected"] = [round(expected.at[r.time, r.zone]) for r in g.itertuples()]
    print("Gap cells by zone:")
    print(g.groupby("Zone").size().sort_values(ascending=False).head(10))
    print("\nGap cells by date:")
    print(g.groupby(g["time"].dt.date).size().sort_values(ascending=False).head(10))

# Preserve observed zeros as ground truth; report suspicious gaps for investigation.
# Only confirmed missing observations should be imputed and excluded from scoring.
final = clean.copy()

# --------- Flag real spikes/drops (kept, not modified) ----------
Lf = np.log1p(final.astype(float))
profile = Lf.loc[train_rows].groupby(train_how).median()
med = pd.DataFrame(profile.reindex(how).to_numpy(), index=Lf.index, columns=Lf.columns)
train_mad = (Lf.loc[train_rows] - med.loc[train_rows]).abs().groupby(train_how).median()
scale = pd.DataFrame((1.4826 * train_mad).clip(lower=0.25).reindex(how).to_numpy(), index=Lf.index, columns=Lf.columns)
rz = (Lf - med) / scale
event_flag = (rz.abs() > 5) & ((final - np.expm1(med)).abs() >= 20) & ~gap

# Cells excluded from evaluation: imputed gaps + DST-adjusted hours
eval_exclude = pd.DataFrame(False, index=clean.index, columns=clean.columns)
eval_exclude.loc[[spring, fall]] = True

# ---------- Validation and summary ----------
assert final.shape == (8760, 263)
assert not final.isna().any().any()
assert (final.to_numpy() >= 0).all()
assert final.index.is_unique and final.index.is_monotonic_increasing

raw_total = int(raw.to_numpy().sum())
phantom_total = int(phantom_report["phantom_removed"].sum())
final_total = int(final.to_numpy().sum())

print("\nSUMMARY")
print("=" * 55)
print(f"Pickups before handling:        {raw_total:,}")
print(f"Phantom records removed:        {phantom_total:,} ({phantom_total / raw_total:.2%})")
print(f"Zero-duration real trips kept:  {int(phantom_report['real_trips_kept'].sum()):,}")
print(f"Suspicious zero cells (preserved):              {int(gap.to_numpy().sum()):,}")
print(f"Event outliers flagged (kept):  {int(event_flag.to_numpy().sum()):,}")
print(f"Cells excluded from evaluation: {int(eval_exclude.to_numpy().sum()):,}")
print(f"Final total pickups:            {final_total:,}")

display(phantom_report)

phantom_report.set_index("month")[["real_trips_kept", "phantom_removed"]].plot(
    kind="bar", stacked=True, figsize=(12, 4),
    title="Zero-Duration Records by Month: Real Trips Kept vs Phantom Records Removed")
plt.ylabel("Records"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

# ---------- Save ----------
final_demand_matrix = final
final.rename(columns=str).to_parquet(SAVE + "demand_2025_clean.parquet")
event_flag.rename(columns=str).to_parquet(SAVE + "event_flags.parquet")
eval_exclude.rename(columns=str).to_parquet(SAVE + "eval_exclude.parquet")
print("\nOUTLIER HANDLING COMPLETED — use demand_2025_clean.parquet from here on.")

In [ ]:
import holidays

print("STEP 18: FEATURE ENGINEERING")
print("=" * 55)

# Load the cleaned demand matrix from Step 17
final_demand_matrix = pd.read_parquet(SAVE + "demand_2025_clean.parquet")
final_demand_matrix.columns = final_demand_matrix.columns.astype(int)
idx = final_demand_matrix.index

# Chronological split boundaries (used everywhere from here on)
TRAIN_END = pd.Timestamp("2025-10-01")   # train: Jan–Sep
VAL_END = pd.Timestamp("2025-11-01")     # validation: Oct | test: Nov–Dec
train_mask = idx < TRAIN_END

weather = pd.read_parquet(SAVE + "weather_2025.parquet").reindex(idx)
dst_flag = pd.read_parquet(SAVE + "dst_flag.parquet")["dst_flag"].reindex(idx).fillna(0)

# Federal holidays (2026 included so Dec 31 gets its pre-holiday flag)
us_hol = holidays.US(years=[2025, 2026])
hol_days = pd.to_datetime(list(us_hol.keys()))
day = idx.normalize()

context = pd.DataFrame(index=idx)

# Cyclical time encodings (hour 23 is next to hour 0; Sunday next to Monday)
context["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
context["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
context["dow_sin"] = np.sin(2 * np.pi * idx.dayofweek / 7)
context["dow_cos"] = np.cos(2 * np.pi * idx.dayofweek / 7)
context["is_weekend"] = (idx.dayofweek >= 5).astype(int)

# Holiday indicators
context["is_holiday"] = day.isin(hol_days).astype(int)
context["pre_holiday"] = (day + pd.Timedelta(days=1)).isin(hol_days).astype(int)
context["post_holiday"] = (day - pd.Timedelta(days=1)).isin(hol_days).astype(int)
context["dst_flag"] = dst_flag.astype(int).to_numpy()

# Weather: standardized with TRAINING statistics only (no leakage)
wx_cols = ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m"]
wx_stats = {}
for c in wx_cols:
    mu, sd = weather.loc[train_mask, c].mean(), weather.loc[train_mask, c].std()
    wx_stats[c] = (mu, sd)
    context[c] = (weather[c] - mu) / sd
context["is_raining"] = (weather["precipitation"] > 0.1).astype(int)

# ---------- Validation ----------
assert len(context) == 8760
assert not context.isna().any().any()
assert context.index.equals(final_demand_matrix.index)

print("Context feature matrix:", context.shape)
print("Features:", context.columns.tolist())

print("\nSplit sizes (hours):")
print("  Train (Jan–Sep):", int(train_mask.sum()))
print("  Validation (Oct):", int(((idx >= TRAIN_END) & (idx < VAL_END)).sum()))
print("  Test (Nov–Dec):", int((idx >= VAL_END).sum()))

print("\nHolidays in the data:")
for d in sorted(hol_days[(hol_days >= idx.min().normalize()) & (hol_days <= idx.max())]):
    split = "train" if d < TRAIN_END else ("val" if d < VAL_END else "test")
    print(f"  {d.date()} ({d.day_name()}): {us_hol.get(d.date())} [{split}]")

print("\nWeather statistics used for scaling (training period):")
display(pd.DataFrame(wx_stats, index=["mean", "std"]).T.round(3))

print("\nFeature summary:")
display(context.describe().T[["mean", "std", "min", "max"]].round(3))

print("\nCorrelation with citywide log demand:")
city_log = np.log1p(final_demand_matrix.sum(axis=1))
print(context.corrwith(city_log).round(3).sort_values())

# Visual check: one sample week
fig, ax = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
wk = slice("2025-07-07", "2025-07-13")
ax[0].plot(city_log.loc[wk]); ax[0].set_title("Citywide log demand (sample week)")
ax[1].plot(context.loc[wk, ["hour_sin", "hour_cos"]]); ax[1].set_title("Cyclical hour encoding")
ax[2].plot(context.loc[wk, ["temperature_2m", "precipitation"]]); ax[2].set_title("Standardized weather")
ax[2].legend(["temperature", "precipitation"])
plt.tight_layout(); plt.show()

context.to_parquet(SAVE + "context_features.parquet")
print("\nFEATURE ENGINEERING COMPLETED!")

In [ ]:
import gc, glob
import geopandas as gpd
from matplotlib.collections import LineCollection

print("GRAPH CONSTRUCTION")
print("=" * 55)

N = len(zone_ids)
K = 8
pos = {int(z): i for i, z in enumerate(zone_ids)}
zone_info = zones.set_index("LocationID").loc[zone_ids, ["Borough", "Zone"]]

# Zone geometry
if "gdf" not in globals():
    !wget -q -nc https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip && unzip -oq taxi_zones.zip -d tz
    gdf = gpd.read_file(glob.glob("tz/**/*.shp", recursive=True)[0]).to_crs(2263)
    gdf = gdf.dissolve(by="LocationID").reindex(zone_ids)
cent = np.c_[gdf.centroid.x, gdf.centroid.y]

# ---------- Contiguity graph  ----------
A_adj = np.load(SAVE + "A_adj.npy")
D = np.load(SAVE + "D_miles.npy")

# --------- Distance graph: Gaussian kernel over K nearest zones ----------
nn = np.argsort(D, axis=1)[:, 1:K + 1]
sigma = np.take_along_axis(D, nn, 1).mean()
rows = np.repeat(np.arange(N), K)
A_dist = np.zeros((N, N))
A_dist[rows, nn.ravel()] = np.exp(-(D[rows, nn.ravel()] / sigma) ** 2)
A_dist = np.maximum(A_dist, A_dist.T)
print(f"Distance kernel bandwidth (sigma): {sigma:.2f} miles")

# ---------- OD-flow graph: training months only ----------
print("\nBuilding OD-flow matrix from training months (Jan–Sep):")
od = np.zeros((N, N))
for month in range(1, 10):
    url = ("https://d37ci6vzurychx.cloudfront.net/"
           f"trip-data/yellow_tripdata_2025-{month:02d}.parquet")
    t = pd.read_parquet(url, columns=["tpep_pickup_datetime", "tpep_dropoff_datetime",
                                      "PULocationID", "DOLocationID"])
    start = pd.Timestamp(2025, month, 1)
    end = start + pd.DateOffset(months=1)
    m = ((t["tpep_pickup_datetime"] >= start) & (t["tpep_pickup_datetime"] < end) &
         (t["tpep_dropoff_datetime"] > t["tpep_pickup_datetime"]) &      # real completed trips
         t["PULocationID"].isin(zone_ids) & t["DOLocationID"].isin(zone_ids))
    g = t.loc[m].groupby(["PULocationID", "DOLocationID"]).size()
    i = g.index.get_level_values(0).map(pos).to_numpy()
    j = g.index.get_level_values(1).map(pos).to_numpy()
    np.add.at(od, (i, j), g.to_numpy())
    print(f"  {start.strftime('%B')}: {int(g.sum()):,} trips")
    del t, g; gc.collect()

print(f"\nTotal OD trips: {int(od.sum()):,}")
print(f"Intra-zone trip share (pickup = dropoff zone): {np.trace(od) / od.sum():.1%}")

flow = od + od.T
np.fill_diagonal(flow, 0)

def top_k_graph(M, k=10):
    """Keep each zone's k strongest links, scale rows to [0, 1], symmetrize."""
    out = np.zeros_like(M, dtype=float)
    top = np.argsort(-M, axis=1)[:, :k]
    r = np.repeat(np.arange(len(M)), k)
    out[r, top.ravel()] = M[r, top.ravel()]
    out[M <= 0] = 0
    out = out / np.maximum(out.max(1, keepdims=True), 1e-9)
    return np.maximum(out, out.T)

A_od = top_k_graph(np.log1p(flow), k=10)

print("\nTop 10 zone pairs by two-way trip flow:")
flat = np.argsort(-np.triu(flow), axis=None)[:10]
for a, b in zip(*np.unravel_index(flat, flow.shape)):
    print(f"  {zone_info.iloc[a]['Zone']} <-> {zone_info.iloc[b]['Zone']}: "
          f"{int(flow[a, b]):,} trips ({D[a, b]:.1f} mi apart)")

# ---------- Correlation graph: residual demand co-movement, training only ----------
train_end = idx.get_loc(TRAIN_END)
Ltr = np.log1p(final_demand_matrix.to_numpy()[:train_end].astype(float))
how_tr = (idx.dayofweek * 24 + idx.hour).to_numpy()[:train_end]
resid = Ltr - pd.DataFrame(Ltr).groupby(how_tr).transform("mean").to_numpy()
with np.errstate(invalid="ignore", divide="ignore"):
    Cr = np.nan_to_num(np.corrcoef(resid.T))
np.fill_diagonal(Cr, 0)
A_corr = top_k_graph(np.clip(Cr, 0, None), k=10)

# ---------- Comparison ----------
graphs = {"Contiguity": A_adj, "Distance (kNN)": A_dist, "OD flow": A_od, "Correlation": A_corr}

def jaccard(a, b):
    a, b = a > 0, b > 0
    return (a & b).sum() / (a | b).sum()

summary = pd.DataFrame({
    name: {"edges": int((A > 0).sum() / 2),
           "mean_degree": round((A > 0).sum(1).mean(), 1),
           "isolated_nodes": int(((A > 0).sum(1) == 0).sum()),
           "mean_edge_length_mi": round(D[A > 0].mean(), 2),
           "overlap_with_contiguity": round(jaccard(A, A_adj), 3)}
    for name, A in graphs.items()}).T
print("\nGraph comparison:")
display(summary)

for name, A in graphs.items():
    assert A.shape == (N, N) and np.allclose(A, A.T) and (np.diag(A) == 0).all()

# ---------- Visualize each graph on the map ----------
fig, axes = plt.subplots(2, 2, figsize=(16, 16))
for ax, (name, A) in zip(axes.ravel(), graphs.items()):
    gdf.boundary.plot(ax=ax, lw=0.2, color="lightgray")
    i, j = np.where(np.triu(A) > 0)
    segs = np.stack([cent[i], cent[j]], axis=1)
    ax.add_collection(LineCollection(segs, lw=0.5, alpha=0.6,
                                     colors=plt.cm.viridis(A[i, j] / A.max())))
    ax.scatter(cent[:, 0], cent[:, 1], s=3, c="black")
    ax.set_title(f"{name} graph ({len(i):,} edges)"); ax.axis("off")
plt.tight_layout(); plt.show()

np.savez(SAVE + "graphs.npz", zone_ids=np.asarray(zone_ids), A_adj=A_adj, A_dist=A_dist, A_od=A_od, A_corr=A_corr, od_raw=od)
print("\nGRAPH CONSTRUCTION COMPLETED!")

In [ ]:
import folium
import branca.colormap as bcm

print("\nINTERACTIVE GRAPH MAP")

# Zone polygons and centroids in lat/lon
zl = gdf[["geometry"]].reset_index(drop=True).copy()
zl["LocationID"] = [int(z) for z in zone_ids]
zl["Zone"] = zone_info["Zone"].values
zl["Borough"] = zone_info["Borough"].values
zl["mean_per_hour"] = final_demand_matrix.mean().round(1).values
short = {"Contiguity": "deg_adj", "Distance (kNN)": "deg_dist",
         "OD flow": "deg_od", "Correlation": "deg_corr"}
for name, A in graphs.items():
    zl[short[name]] = (A > 0).sum(1).astype(int)
zl = zl.to_crs(4326)

cent_ll = gdf.centroid.to_crs(4326)
lat, lon = cent_ll.y.values, cent_ll.x.values

m = folium.Map(location=[40.73, -73.94], zoom_start=11, tiles="cartodbpositron")

# Zone layer coloured by mean demand (log scale)
cmap = bcm.linear.YlOrRd_09.scale(0, np.log1p(zl["mean_per_hour"].max()))
cmap.caption = "log(1 + mean pickups per hour)"
folium.GeoJson(
    zl,
    name="Taxi zones (mean demand)",
    style_function=lambda f: {
        "fillColor": cmap(np.log1p(f["properties"]["mean_per_hour"])),
        "color": "#555", "weight": 0.4, "fillOpacity": 0.45},
    highlight_function=lambda f: {"weight": 2, "color": "black", "fillOpacity": 0.7},
    tooltip=folium.GeoJsonTooltip(
        fields=["LocationID", "Zone", "Borough", "mean_per_hour",
                "deg_adj", "deg_dist", "deg_od", "deg_corr"],
        aliases=["Zone ID", "Zone", "Borough", "Mean pickups/hr",
                 "Contiguity links", "Distance links", "OD-flow links", "Correlation links"]),
).add_to(m)
cmap.add_to(m)

# One toggleable layer per graph
colors = {"Contiguity": "#1f77b4", "Distance (kNN)": "#2ca02c",
          "OD flow": "#d62728", "Correlation": "#9467bd"}
for k, (name, A) in enumerate(graphs.items()):
    fg = folium.FeatureGroup(name=f"{name} edges", show=(k == 0))
    i, j = np.where(np.triu(A) > 0)
    for a, b in zip(i, j):
        tip = (f"<b>{zone_info.iloc[a]['Zone']} ↔ {zone_info.iloc[b]['Zone']}</b><br>"
               f"Edge weight: {A[a, b]:.2f}<br>Distance: {D[a, b]:.1f} mi")
        if name == "OD flow":
            tip += f"<br>Trips (Jan–Sep): {int(flow[a, b]):,}"
        folium.PolyLine(
            [(lat[a], lon[a]), (lat[b], lon[b])],
            color=colors[name], weight=1 + 3 * A[a, b] / A.max(), opacity=0.65,
            tooltip=tip).add_to(fg)
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save(SAVE + "zone_graphs_interactive.html")
display(m)

In [ ]:
import os

print("STEP 20: DATA SPLIT, FORECASTING WINDOWS, AND EVALUATION FRAMEWORK")
print("=" * 65)

# ---------- Load cleaned inputs ----------
final_demand_matrix = pd.read_parquet(SAVE + "demand_2025_clean.parquet")
final_demand_matrix.columns = final_demand_matrix.columns.astype(int)
context = pd.read_parquet(SAVE + "context_features.parquet")
eval_exclude = pd.read_parquet(SAVE + "eval_exclude.parquet").to_numpy()
event_flags = pd.read_parquet(SAVE + "event_flags.parquet").to_numpy()

idx = final_demand_matrix.index
counts = final_demand_matrix.to_numpy().astype(np.float32)   # [T, N] pickups
L = np.log1p(counts)                                         # log scale for modelling
T, N = counts.shape

TRAIN_END = pd.Timestamp("2025-10-01")
VAL_END = pd.Timestamp("2025-11-01")
train_end = idx.get_loc(TRAIN_END)
val_end = idx.get_loc(VAL_END)

# ---------- Forecasting setup ----------
T_IN = 12            # input window (hours)
H = 6                # maximum forecast horizon (hours)
EVAL_H = [1, 3, 6]   # horizons reported
MIN_HISTORY = 168    # one week, so weekly lags exist for every sample

origins = np.arange(MIN_HISTORY - 1, T - H)
train_o = origins[origins + H < train_end]
val_o = origins[(origins + 1 >= train_end) & (origins + H < val_end)]
test_o = origins[origins + 1 >= val_end]

# No target may cross a split boundary
assert (train_o + H < train_end).all()
assert ((val_o + 1 >= train_end) & (val_o + H < val_end)).all()
assert (test_o + 1 >= val_end).all() and (test_o + H < T).all()

split_info = pd.DataFrame({
    name: {"samples": len(o),
           "first_input": idx[o[0] - T_IN + 1],
           "first_target": idx[o[0] + 1],
           "last_target": idx[o[-1] + H]}
    for name, o in [("train", train_o), ("validation", val_o), ("test", test_o)]}).T
print("Forecast samples per split:")
display(split_info)

# ---------- Scaling: per-zone statistics from TRAINING data only ----------
mu = L[:train_end].mean(0)
sd = L[:train_end].std(0)
sd = np.where(sd < 1e-3, 1.0, sd)        # always-zero zones: avoid divide-by-zero
L_scaled = (L - mu) / sd

active = counts[:train_end].mean(0) >= 1
print(f"\nActive zones (training mean ≥ 1 pickup/hr): {active.sum()} of {N}")
print(f"Share of training demand from active zones: "
      f"{counts[:train_end, active].sum() / counts[:train_end].sum():.2%}")

test_cells = eval_exclude[val_end:]
print(f"Test cells excluded from evaluation: {test_cells.sum():,} ({test_cells.mean():.3%})")
print(f"Test cells flagged as events:        {event_flags[val_end:].sum():,}")
print(f"Share of zero-demand test cells:     {(counts[val_end:] == 0).mean():.1%}")

# ---------- Shared evaluation function ----------
RESULTS_PATH = SAVE + "model_results.csv"
results = pd.read_csv(RESULTS_PATH) if os.path.exists(RESULTS_PATH) else pd.DataFrame()

def evaluate(preds, orig, model, split="test", save=True):
    """
    preds: dict {horizon: array [samples, N]} of predicted PICKUP COUNTS
    orig:  forecast origins the predictions belong to
    """
    global results
    rows = []
    for h in EVAL_H:
        raw_prediction = np.asarray(preds[h], dtype=np.float64)
        if raw_prediction.shape != (len(orig), N) or not np.isfinite(raw_prediction).all():
            raise ValueError("Invalid prediction shape or nonfinite values")
        P = np.clip(raw_prediction, 0, None)
        tgt = orig + h
        Y = counts[tgt].astype(np.float64)
        valid = ~eval_exclude[tgt]
        subsets = {"all": valid,
                   "active": valid & active[None, :],
                   "events": valid & event_flags[tgt]}
        for name, msk in subsets.items():
            if msk.sum() == 0:
                continue
            y, p = Y[msk], P[msk]
            e = p - y
            rows.append(dict(model=model, split=split, horizon=h, subset=name,
                             cells=int(msk.sum()),
                             MAE=np.abs(e).mean(),
                             RMSE=np.sqrt((e ** 2).mean()),
                             WMAPE=np.abs(e).sum() / y.sum() if y.sum() > 0 else np.nan))
    new = pd.DataFrame(rows)
    if save:
        if len(results):   # replace earlier runs of the same model/split
            results = results[~((results["model"] == model) & (results["split"] == split))]
        results = pd.concat([results, new], ignore_index=True)
        results.to_csv(RESULTS_PATH, index=False)
    return new

def show(split="test", subset="all"):
    r = results[(results["split"] == split) & (results["subset"] == subset)]
    t = r.pivot_table(index="model", columns="horizon", values=["MAE", "RMSE", "WMAPE"])
    return t.sort_values(("MAE", EVAL_H[0])).round(3)

# ---------- Sanity check: perfect predictions must score zero ----------
perfect = {h: counts[test_o + h] for h in EVAL_H}
chk = evaluate(perfect, test_o, "_perfect_check", save=False)
assert np.allclose(chk[["MAE", "RMSE", "WMAPE"]].to_numpy(), 0)
print("\nEvaluation function sanity check passed (perfect forecast → zero error).")

# ---------- Example model input shapes ----------
o = train_o[0]
x_demand = L_scaled[o - T_IN + 1: o + 1]              # [T_IN, N]
x_context = context.to_numpy()[o - T_IN + 1: o + 1]    # [T_IN, n_context]
y_target = L[o + 1: o + H + 1]                          # [H, N]
print(f"\nExample sample — demand input {x_demand.shape}, "
      f"context input {x_context.shape}, target {y_target.shape}")

# ---------- Visualize the split ----------
daily = final_demand_matrix.sum(axis=1).resample("D").sum()
ax = daily.plot(figsize=(14, 4), color="black", lw=1,
                title="Chronological Train / Validation / Test Split")
ax.axvspan(idx[0], TRAIN_END, alpha=0.15, color="tab:blue", label="Train (Jan–Sep)")
ax.axvspan(TRAIN_END, VAL_END, alpha=0.15, color="tab:orange", label="Validation (Oct)")
ax.axvspan(VAL_END, idx[-1], alpha=0.15, color="tab:green", label="Test (Nov–Dec)")
ax.set_ylabel("Pickups per day"); ax.legend(loc="lower left"); plt.show()

np.savez(SAVE + "prep.npz", train_o=train_o, val_o=val_o, test_o=test_o,
         mu=mu, sd=sd, active=active)
print("\nSPLIT AND EVALUATION FRAMEWORK READY!")

In [ ]:
print("SIMPLE BASELINE MODELS")
print("=" * 55)

how = (idx.dayofweek * 24 + idx.hour).to_numpy()   # hour-of-week, 0..167
zone_list = final_demand_matrix.columns.tolist()

# Historical average table from TRAINING data only: [168 hour-of-week, N zones]
ha_table = np.stack([counts[:train_end][how[:train_end] == k].mean(0) for k in range(168)])

def baseline_predictions(orig):
    assert orig.min() + 1 - 4 * 168 >= 0, "Not enough history for the 4-week average"
    return {
        "Persistence (last hour)":
            {h: counts[orig] for h in EVAL_H},
        "Seasonal naive (same hour yesterday)":
            {h: counts[orig + h - 24] for h in EVAL_H},
        "Seasonal naive (same hour last week)":
            {h: counts[orig + h - 168] for h in EVAL_H},
        "Historical average (hour-of-week)":
            {h: ha_table[how[orig + h]] for h in EVAL_H},
        "Weekly moving average (last 4 weeks)":
            {h: np.mean([counts[orig + h - 168 * k] for k in range(1, 5)], axis=0)
             for h in EVAL_H},
    }

for split, orig in [("val", val_o), ("test", test_o)]:
    for name, preds in baseline_predictions(orig).items():
        evaluate(preds, orig, name, split=split)
print("Baselines evaluated on validation and test sets.")

# ---------- Results ----------
print("\nTEST SET — ALL ZONES")
display(show("test", "all"))
print("\nTEST SET — ACTIVE ZONES")
display(show("test", "active"))
print("\nTEST SET — EVENT CELLS")
display(show("test", "events"))
print("\nVALIDATION SET — ALL ZONES")
display(show("val", "all"))

# ---------- Demand level check (explains historical-average bias) ----------
city_train = counts[:train_end].sum(1).mean()
city_test = counts[val_end:].sum(1).mean()
print(f"\nMean citywide pickups/hour — train: {city_train:,.0f} | test: {city_test:,.0f} "
      f"({city_test / city_train - 1:+.1%})")

# ---------- Error growth with horizon ----------
r = results[(results["split"] == "test") & (results["subset"] == "all")]
r.pivot_table(index="horizon", columns="model", values="MAE").plot(
    marker="o", figsize=(10, 4), title="Baseline MAE by Forecast Horizon (Test, All Zones)")
plt.xlabel("Forecast horizon (hours)"); plt.ylabel("MAE (pickups)"); plt.xticks(EVAL_H)
plt.legend(fontsize=8); plt.show()

# ---------- Visual check: first test week, 1-hour-ahead forecasts ----------
preds_test = baseline_predictions(test_o)
week = test_o[:168]
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, z in zip(axes, [161, 132]):
    j = zone_list.index(z)
    ax.plot(idx[week + 1], counts[week + 1, j], color="black", lw=1.5, label="Actual")
    for name in ["Persistence (last hour)", "Historical average (hour-of-week)",
                 "Weekly moving average (last 4 weeks)"]:
        ax.plot(idx[week + 1], preds_test[name][1][:168, j], lw=1, alpha=0.8, label=name)
    ax.set_title(f"{zone_info.loc[z, 'Zone']} — 1-hour-ahead baseline forecasts (first test week)")
    ax.set_ylabel("Pickups")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("\nBASELINE EVALUATION COMPLETED!")

In [ ]:
import lightgbm as lgb, time, gc

print("LIGHTGBM MODELS")
print("=" * 55)

# ---------- Inputs ----------
G = np.load(SAVE + "graphs.npz")

def row_norm(A):
    d = A.sum(1, keepdims=True)
    return np.divide(A, d, out=np.zeros_like(A), where=d > 0)

W_adj = row_norm(G["A_adj"]).astype(np.float32)   # contiguity neighbours
W_od = row_norm(G["A_od"]).astype(np.float32)     # OD-flow neighbours

cal_cols = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend",
            "is_holiday", "pre_holiday", "post_holiday", "dst_flag"]
wx_cols = ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m", "is_raining"]
CAL = context[cal_cols].to_numpy(np.float32)
WX = context[wx_cols].to_numpy(np.float32)
zone_mean = L[:train_end].mean(0)

# Cumulative sums for fast rolling means
cs = np.vstack([np.zeros((1, N), np.float32), np.cumsum(L, axis=0)])
def roll(o, k):
    return (cs[o + 1] - cs[o + 1 - k]) / k

def make_features(orig, h, spatial):
    """One row per (origin, zone). Only information available at the origin is used,
    except calendar features of the target hour, which are known in advance."""
    tgt = orig + h
    S = len(orig)
    f = {
        "lag_0": L[orig], "lag_1": L[orig - 1], "lag_2": L[orig - 2], "lag_3": L[orig - 3],
        "lag_tgt_24": L[tgt - 24],        # same hour yesterday (relative to target)
        "lag_tgt_168": L[tgt - 168],      # same hour last week
        "roll_3": roll(orig, 3), "roll_24": roll(orig, 24), "roll_168": roll(orig, 168),
        "zone_mean": np.broadcast_to(zone_mean, (S, N)),
        "zone_id": np.broadcast_to(np.arange(N), (S, N)),
    }
    for k, c in enumerate(cal_cols):                       # calendar of target hour
        f[c] = np.broadcast_to(CAL[tgt, k][:, None], (S, N))
    for k, c in enumerate(wx_cols):                        # weather at origin (no future info)
        f[c] = np.broadcast_to(WX[orig, k][:, None], (S, N))
    if spatial:
        for gname, W in [("adj", W_adj), ("od", W_od)]:
            f[f"nbr_{gname}_lag_0"] = L[orig] @ W.T
            f[f"nbr_{gname}_roll_3"] = roll(orig, 3) @ W.T
            f[f"nbr_{gname}_lag_tgt_168"] = L[tgt - 168] @ W.T
    X = pd.DataFrame({k: np.ascontiguousarray(v, dtype=np.float32).ravel() for k, v in f.items()})
    X["zone_id"] = X["zone_id"].astype(np.int32)
    y = counts[tgt].ravel()
    return X, y

params = dict(objective="poisson", metric="l1", learning_rate=0.1,
              num_leaves=127, min_data_in_leaf=200, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              seed=42, verbose=-1, num_threads=os.cpu_count())

variants = {"LightGBM (no spatial features)": False,
            "LightGBM + spatial lags": True}
importances = {}
full_o = np.concatenate([train_o, val_o, test_o])     # every forecast origin in 2025
full_preds = {}

# ---------- Train on Jan–Sep, predict the whole year ----------
for name, spatial in variants.items():
    preds = {"train": {}, "val": {}, "test": {}}
    for h in EVAL_H:
        t0 = time.time()
        Xtr, ytr = make_features(train_o, h, spatial)
        Xva, yva = make_features(val_o, h, spatial)
        Xte, _ = make_features(test_o, h, spatial)
        ktr = ~eval_exclude[train_o + h].ravel()     # drop imputed/DST targets
        kva = ~eval_exclude[val_o + h].ravel()

        dtr = lgb.Dataset(Xtr.loc[ktr], ytr[ktr], categorical_feature=["zone_id"])
        dva = lgb.Dataset(Xva.loc[kva], yva[kva], reference=dtr)
        model = lgb.train(params, dtr, num_boost_round=1500, valid_sets=[dva],
                          callbacks=[lgb.early_stopping(50, verbose=False)])

        it = model.best_iteration
        preds["train"][h] = model.predict(Xtr, num_iteration=it).reshape(len(train_o), N)
        preds["val"][h] = model.predict(Xva, num_iteration=it).reshape(len(val_o), N)
        preds["test"][h] = model.predict(Xte, num_iteration=it).reshape(len(test_o), N)
        importances[(name, h)] = pd.Series(model.feature_importance("gain"), index=Xtr.columns)

        tag = "spatial" if spatial else "base"
        model.save_model(SAVE + f"lgbm_{tag}_h{h}.txt")
        print(f"{name} | h={h}: {it} trees, {Xtr.shape[1]} features, {time.time() - t0:.0f}s")
        del Xtr, ytr, Xva, yva, Xte, dtr, dva, model; gc.collect()

    # Full-year predictions = train + val + test, in chronological order
    preds["full"] = {h: np.concatenate([preds[s][h] for s in ["train", "val", "test"]])
                     for h in EVAL_H}
    for split, orig in [("train", train_o), ("val", val_o), ("test", test_o), ("full", full_o)]:
        evaluate(preds[split], orig, name, split=split)

    full_preds[name] = preds["full"]
    tag = "spatial" if spatial else "base"
    np.savez(SAVE + f"preds_lgbm_{tag}_full.npz", **{f"h{h}": preds["full"][h] for h in EVAL_H})

# ---------- Results across the whole dataset ----------
split_order = ["train", "val", "test", "full"]
split_names = {"train": "Train (Jan–Sep, in-sample)", "val": "Validation (Oct)",
               "test": "Test (Nov–Dec)", "full": "Full year 2025"}

def split_summary(models, subset="all", metric="MAE"):
    r = results[results["model"].isin(models) & (results["subset"] == subset)]
    t = r.pivot_table(index=["model", "split"], columns="horizon", values=metric)
    t = t.reindex([(m, s) for m in models for s in split_order]).rename(index=split_names, level=1)
    return t.round(3)

lgb_models = list(variants)
for subset in ["all", "active", "events"]:
    print(f"\nMAE BY SPLIT — {subset.upper()} ZONES" if subset != "events" else "\nMAE BY SPLIT — EVENT CELLS")
    display(split_summary(lgb_models, subset, "MAE"))
print("\nWMAPE BY SPLIT — ALL ZONES")
display(split_summary(lgb_models, "all", "WMAPE"))

# ---------- Overfitting check ----------
r = results[results["subset"] == "all"].set_index(["model", "split", "horizon"])["MAE"]
print("\nGeneralization gap (test MAE / train MAE):")
for m in lgb_models:
    print(f"  {m}: " + " | ".join(f"h={h}: {r[(m, 'test', h)] / r[(m, 'train', h)]:.2f}x"
                                    for h in EVAL_H))

# ---------- Does spatial information help? (every split) ----------
print("\nEffect of spatial lags on MAE (negative = improvement):")
for s in split_order:
    print(f"  {split_names[s]}: " + " | ".join(
        f"h={h}: {(r[('LightGBM + spatial lags', s, h)] / r[('LightGBM (no spatial features)', s, h)] - 1) * 100:+.1f}%"
        for h in EVAL_H))

# ---------- Full-year visual: actual vs predicted citywide demand (h = 1) ----------
actual_city = pd.Series(counts[full_o + 1].sum(1), index=idx[full_o + 1]).resample("D").sum()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(actual_city, color="black", lw=1.5, label="Actual")
for m in lgb_models:
    pred_city = pd.Series(full_preds[m][1].sum(1), index=idx[full_o + 1]).resample("D").sum()
    ax.plot(pred_city, lw=1, alpha=0.8, label=m)
ax.axvspan(idx[0], TRAIN_END, alpha=0.08, color="tab:blue", label="Train")
ax.axvspan(TRAIN_END, VAL_END, alpha=0.12, color="tab:orange", label="Validation")
ax.axvspan(VAL_END, idx[-1], alpha=0.12, color="tab:green", label="Test")
ax.set_title("Daily Citywide Pickups: Actual vs LightGBM 1-Hour-Ahead Forecasts, Full Year 2025")
ax.set_ylabel("Pickups per day"); ax.legend(fontsize=8, ncol=3); plt.show()

# ---------- Monthly MAE across the year (h = 1, all zones) ----------
monthly = {}
tgt1 = full_o + 1
valid = ~eval_exclude[tgt1]
for m in lgb_models:
    err = np.abs(np.clip(full_preds[m][1], 0, None) - counts[tgt1])
    err = np.where(valid, err, np.nan)
    monthly[m] = pd.Series(np.nanmean(err, axis=1), index=idx[tgt1]).resample("ME").mean()
pd.DataFrame(monthly).plot(kind="bar", figsize=(12, 4),
    title="Monthly MAE Across 2025 (1-Hour-Ahead, All Zones)")
plt.xticks(range(12), [d.strftime("%b") for d in pd.DataFrame(monthly).index], rotation=0)
plt.ylabel("MAE (pickups)"); plt.axvline(8.5, ls="--", c="gray"); plt.axvline(9.5, ls="--", c="gray")
plt.show()

# ---------- Feature importance ----------
imp = importances[("LightGBM + spatial lags", 1)]
imp = (imp / imp.sum()).sort_values(ascending=True).tail(15)
imp.plot(kind="barh", figsize=(9, 6),
         title="LightGBM + Spatial Lags — Top 15 Features (h = 1, share of total gain)")
plt.xlabel("Share of total gain"); plt.tight_layout(); plt.show()

print("\nLIGHTGBM EVALUATION COMPLETED!")

In [ ]:
import torch, torch.nn as nn, time, copy, re

print("STEP 23: DEEP LEARNING BASELINE — GRU WITHOUT GRAPH")
print("=" * 55)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
torch.manual_seed(42); np.random.seed(42)

# ---------- Tensors on GPU ----------
cal_cols = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend",
            "is_holiday", "pre_holiday", "post_holiday", "dst_flag"]
wx_cols = ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m", "is_raining"]
ctx_cols = cal_cols + wx_cols

Ls_t = torch.tensor(L_scaled, dtype=torch.float32, device=device)            # [T, N]
C_t = torch.tensor(context[ctx_cols].to_numpy(np.float32), device=device)   # [T, F]
EX_t = torch.tensor(eval_exclude, device=device)                            # [T, N] bool
mu_t = torch.tensor(mu, dtype=torch.float32, device=device)
sd_t = torch.tensor(sd, dtype=torch.float32, device=device)
win_off = torch.arange(-T_IN + 1, 1, device=device)
tgt_off = torch.arange(1, H + 1, device=device)
N_CTX = len(ctx_cols)

full_o = np.concatenate([train_o, val_o, test_o])      # every forecast origin in 2025
full_preds = globals().get("full_preds", {})            # keeps LightGBM predictions if present

def get_batch(orig):
    """orig: LongTensor of forecast origins -> inputs, targets, valid-target mask."""
    win = orig[:, None] + win_off            # [B, T_IN]
    tgt = orig[:, None] + tgt_off            # [B, H]
    return Ls_t[win], C_t[win], Ls_t[tgt], ~EX_t[tgt]

# ---------- Shared training / prediction utilities (reused in Step 24) ----------
@torch.no_grad()
def eval_loss(model, orig_t, batch_size=64):
    model.eval(); num = den = 0.0
    for i in range(0, len(orig_t), batch_size):
        xd, xc, y, m = get_batch(orig_t[i:i + batch_size])
        num += ((model(xd, xc) - y).abs() * m).sum().item()
        den += m.sum().item()
    return num / den

def train_model(model, name, epochs=60, batch_size=32, lr=1e-3, patience=6):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    tr = torch.tensor(train_o, device=device)
    va = torch.tensor(val_o, device=device)
    best, best_state, wait, hist = np.inf, None, 0, []
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTraining {name} ({n_params:,} parameters)")

    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); tl, nb = 0.0, 0
        perm = tr[torch.randperm(len(tr), device=device)]
        for i in range(0, len(perm), batch_size):
            xd, xc, y, m = get_batch(perm[i:i + batch_size])
            loss = ((model(xd, xc) - y).abs() * m).sum() / m.sum()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tl += loss.item(); nb += 1
        vl = eval_loss(model, va)
        sched.step(vl)
        hist.append((ep, tl / nb, vl))
        print(f"  epoch {ep:2d} | train {tl / nb:.4f} | val {vl:.4f} | "
              f"lr {opt.param_groups[0]['lr']:.1e} | {time.time() - t0:.0f}s")
        if vl < best - 1e-4:
            best, best_state, wait = vl, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  early stopping at epoch {ep} (best val {best:.4f})")
                break

    model.load_state_dict(best_state)
    slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    torch.save(best_state, SAVE + f"model_{slug}.pt")
    return model, pd.DataFrame(hist, columns=["epoch", "train_loss", "val_loss"])

@torch.no_grad()
def predict_counts(model, orig, batch_size=64):
    model.eval(); out = []
    o_t = torch.tensor(orig, device=device)
    for i in range(0, len(o_t), batch_size):
        xd, xc, _, _ = get_batch(o_t[i:i + batch_size])
        logp = model(xd, xc) * sd_t + mu_t                     # back to log1p scale
        out.append(torch.expm1(logp).clamp_min(0).cpu().numpy())
    P = np.concatenate(out)                                    # [S, H, N]
    return {h: P[:, h - 1, :] for h in EVAL_H}

def fit_and_evaluate(model, name, **kw):
    """Train on Jan–Sep (early stopping on Oct), then predict and evaluate the whole year."""
    model, hist = train_model(model, name, **kw)
    hist.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", figsize=(8, 3),
              title=f"{name} — Training Curve")
    plt.ylabel("Masked MAE (scaled log demand)"); plt.show()

    preds = {s: predict_counts(model, o) for s, o in
             [("train", train_o), ("val", val_o), ("test", test_o)]}
    preds["full"] = {h: np.concatenate([preds[s][h] for s in ["train", "val", "test"]])
                     for h in EVAL_H}
    for s, o in [("train", train_o), ("val", val_o), ("test", test_o), ("full", full_o)]:
        evaluate(preds[s], o, name, split=s)

    full_preds[name] = preds["full"]
    slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    np.savez(SAVE + f"preds_{slug}_full.npz", **{f"h{h}": preds["full"][h] for h in EVAL_H})
    return model, hist

# ---------- Shared reporting utilities (reused in Step 24) ----------
split_order = ["train", "val", "test", "full"]
split_names = {"train": "Train (Jan–Sep, in-sample)", "val": "Validation (Oct)",
               "test": "Test (Nov–Dec)", "full": "Full year 2025"}

def split_summary(models, subset="all", metric="MAE"):
    models = [m for m in models if m in set(results["model"])]
    r = results[results["model"].isin(models) & (results["subset"] == subset)]
    t = r.pivot_table(index=["model", "split"], columns="horizon", values=metric)
    t = t.reindex([(m, s) for m in models for s in split_order]).rename(index=split_names, level=1)
    return t.round(3)

def generalization_gap(models):
    r = results[results["subset"] == "all"].set_index(["model", "split", "horizon"])["MAE"]
    print("\nGeneralization gap (test MAE / train MAE):")
    for m in models:
        if (m, "train", EVAL_H[0]) in r.index:
            print(f"  {m}: " + " | ".join(
                f"h={h}: {r[(m, 'test', h)] / r[(m, 'train', h)]:.2f}x" for h in EVAL_H))

def plot_full_year(models, h=1):
    models = [m for m in models if m in full_preds]
    tgt = full_o + h
    actual = pd.Series(counts[tgt].sum(1), index=idx[tgt]).resample("D").sum()
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(actual, color="black", lw=1.5, label="Actual")
    for m in models:
        ax.plot(pd.Series(full_preds[m][h].sum(1), index=idx[tgt]).resample("D").sum(),
                lw=1, alpha=0.8, label=m)
    ax.axvspan(idx[0], TRAIN_END, alpha=0.08, color="tab:blue", label="Train")
    ax.axvspan(TRAIN_END, VAL_END, alpha=0.12, color="tab:orange", label="Validation")
    ax.axvspan(VAL_END, idx[-1], alpha=0.12, color="tab:green", label="Test")
    ax.set_title(f"Daily Citywide Pickups: Actual vs {h}-Hour-Ahead Forecasts, Full Year 2025")
    ax.set_ylabel("Pickups per day"); ax.legend(fontsize=8, ncol=3); plt.show()

def plot_monthly_mae(models, h=1):
    models = [m for m in models if m in full_preds]
    tgt = full_o + h
    valid = ~eval_exclude[tgt]
    monthly = {}
    for m in models:
        err = np.abs(np.clip(full_preds[m][h], 0, None) - counts[tgt])
        err = np.where(valid, err, np.nan)
        monthly[m] = pd.Series(np.nanmean(err, axis=1), index=idx[tgt]).resample("ME").mean()
    mdf = pd.DataFrame(monthly)
    ax = mdf.plot(kind="bar", figsize=(12, 4),
                  title=f"Monthly MAE Across 2025 ({h}-Hour-Ahead, All Zones)")
    ax.set_xticks(range(len(mdf))); ax.set_xticklabels([d.strftime("%b") for d in mdf.index], rotation=0)
    ax.axvline(8.5, ls="--", c="gray"); ax.axvline(9.5, ls="--", c="gray")
    ax.set_ylabel("MAE (pickups)"); ax.legend(fontsize=8); plt.show()
    return mdf.round(3)

# ---------- Model: one shared GRU applied to every zone independently ----------
class NodeGRU(nn.Module):
    def __init__(self, n_nodes, n_ctx, hidden=64, emb=16, layers=2, horizon=H, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, emb)
        self.gru = nn.GRU(1 + n_ctx + emb, hidden, layers, batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, horizon))

    def forward(self, x_dem, x_ctx):            # x_dem [B,T,N], x_ctx [B,T,F]
        B, T, N = x_dem.shape
        e = self.emb.weight[None, None].expand(B, T, N, -1)
        c = x_ctx[:, :, None, :].expand(B, T, N, -1)
        z = torch.cat([x_dem[..., None], c, e], dim=-1)          # [B,T,N,D]
        z = z.permute(0, 2, 1, 3).reshape(B * N, T, -1)          # each zone = one sequence
        out, _ = self.gru(z)
        y = self.head(out[:, -1])                                # [B*N, H]
        return y.view(B, N, -1).permute(0, 2, 1)                 # [B, H, N]

gru_model, gru_hist = fit_and_evaluate(NodeGRU(N, N_CTX), "GRU (no graph)")

# ---------- Results across the whole dataset ----------
compare = ["GRU (no graph)", "LightGBM (no spatial features)", "LightGBM + spatial lags"]

print("\nMAE BY SPLIT — ALL ZONES")
display(split_summary(compare, "all", "MAE"))
print("\nMAE BY SPLIT — ACTIVE ZONES")
display(split_summary(compare, "active", "MAE"))
print("\nMAE BY SPLIT — EVENT CELLS")
display(split_summary(compare, "events", "MAE"))
print("\nWMAPE BY SPLIT — ALL ZONES")
display(split_summary(compare, "all", "WMAPE"))

generalization_gap(compare)

r = results[results["subset"] == "all"].set_index(["model", "split", "horizon"])["MAE"]
print("\nGRU vs LightGBM — MAE on every split (all zones):")
for s in split_order:
    print(f"  {split_names[s]}:")
    for h in EVAL_H:
        vals = " | ".join(f"{m.split(' (')[0] if m.startswith('GRU') else m}: {r[(m, s, h)]:.3f}"
                          for m in compare if (m, s, h) in r.index)
        print(f"    h={h}: {vals}")

plot_full_year(compare, h=1)
print("Monthly MAE (1-hour-ahead, all zones):")
display(plot_monthly_mae(compare, h=1))

print("\nGRU BASELINE COMPLETED!")

In [ ]:
import torch.nn.functional as F

print("GRAPH NEURAL NETWORK MODELS")
print("=" * 55)

G = np.load(SAVE + "graphs.npz")

def sym_norm(A):
    """GCN normalization with self-loops: D^-1/2 (A + I) D^-1/2."""
    A = A + np.eye(len(A))
    d = 1.0 / np.sqrt(A.sum(1))
    return (A * d[:, None]) * d[None, :]

def row_norm(A):
    d = A.sum(1, keepdims=True)
    return np.divide(A, d, out=np.zeros_like(A, dtype=float), where=d > 0)

# Random control graph: same degree distribution as contiguity, edges assigned to random zones
rng = np.random.default_rng(42)
perm = rng.permutation(N)
A_rand = G["A_adj"][perm][:, perm]

graph_set = {
    "contiguity": G["A_adj"],
    "distance": G["A_dist"],
    "OD flow": G["A_od"],
    "correlation": G["A_corr"],
    "random (control)": A_rand,
}

# ---------- Model 1: GCN-GRU ----------
class GraphConv(nn.Module):
    """K-hop graph convolution: concatenates [X, AX, A²X, ...] then projects."""
    def __init__(self, d_in, d_out, A_hat, K=2):
        super().__init__()
        self.register_buffer("A", torch.tensor(A_hat, dtype=torch.float32))
        self.K = K
        self.lin = nn.Linear(d_in * (K + 1), d_out)

    def forward(self, x):                          # x [..., N, D]
        out, h = [x], x
        for _ in range(self.K):
            h = torch.einsum("nm,...md->...nd", self.A, h)
            out.append(h)
        return self.lin(torch.cat(out, dim=-1))

class GCNGRU(nn.Module):
    def __init__(self, n_nodes, n_ctx, A_hat, hidden=64, emb=16, layers=2,
                 horizon=H, dropout=0.1, K=2):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, emb)
        self.inp = nn.Linear(1 + n_ctx + emb, hidden)
        self.gc_in = GraphConv(hidden, hidden, A_hat, K)       # spatial mixing per hour
        self.gru = nn.GRU(hidden, hidden, layers, batch_first=True, dropout=dropout)
        self.gc_out = GraphConv(hidden, hidden, A_hat, K)      # spatial mixing of temporal state
        self.norm = nn.LayerNorm(hidden)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, horizon))

    def forward(self, x_dem, x_ctx):               # x_dem [B,T,N], x_ctx [B,T,F]
        B, T, N = x_dem.shape
        e = self.emb.weight[None, None].expand(B, T, N, -1)
        c = x_ctx[:, :, None, :].expand(B, T, N, -1)
        h = torch.relu(self.inp(torch.cat([x_dem[..., None], c, e], dim=-1)))   # [B,T,N,D]
        h = h + torch.relu(self.gc_in(h))
        h = h.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.gru(h)
        s = out[:, -1].view(B, N, -1)
        s = self.norm(s + torch.relu(self.gc_out(s)))
        return self.head(s).permute(0, 2, 1)       # [B,H,N]

# ---------- Model 2: Graph WaveNet ----------
class DiffusionConv(nn.Module):
    def __init__(self, c_in, c_out, n_supports, order=2, dropout=0.3):
        super().__init__()
        self.order = order
        self.mlp = nn.Conv2d(c_in * (1 + order * n_supports), c_out, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, supports):                # x [B,C,N,T]
        out = [x]
        for A in supports:
            x1 = torch.einsum("bcvt,wv->bcwt", x, A)
            out.append(x1)
            for _ in range(self.order - 1):
                x1 = torch.einsum("bcvt,wv->bcwt", x1, A)
                out.append(x1)
        return self.dropout(self.mlp(torch.cat(out, dim=1)))

class GraphWaveNet(nn.Module):
    def __init__(self, n_nodes, n_ctx, supports, adaptive=True, emb=16, res_ch=32,
                 dil_ch=32, skip_ch=128, end_ch=256, horizon=H, blocks=4, layers=2,
                 dropout=0.3, apt_dim=10):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, emb)
        self.n_fixed = len(supports)
        for i, s in enumerate(supports):
            self.register_buffer(f"S{i}", torch.tensor(s, dtype=torch.float32))
        self.adaptive = adaptive
        if adaptive:
            self.nodevec1 = nn.Parameter(torch.randn(n_nodes, apt_dim) * 0.1)
            self.nodevec2 = nn.Parameter(torch.randn(apt_dim, n_nodes) * 0.1)
        n_sup = self.n_fixed + int(adaptive)

        self.start = nn.Conv2d(1 + n_ctx + emb, res_ch, 1)
        self.filt, self.gate, self.skip, self.gconv, self.bn = (nn.ModuleList() for _ in range(5))
        rf = 1
        for _ in range(blocks):
            d = 1
            for _ in range(layers):
                self.filt.append(nn.Conv2d(res_ch, dil_ch, (1, 2), dilation=(1, d)))
                self.gate.append(nn.Conv2d(res_ch, dil_ch, (1, 2), dilation=(1, d)))
                self.skip.append(nn.Conv2d(dil_ch, skip_ch, 1))
                self.gconv.append(DiffusionConv(dil_ch, res_ch, n_sup, dropout=dropout))
                self.bn.append(nn.BatchNorm2d(res_ch))
                rf += d
                d *= 2
        self.receptive_field = rf
        self.end1 = nn.Conv2d(skip_ch, end_ch, 1)
        self.end2 = nn.Conv2d(end_ch, horizon, 1)

    def adaptive_adj(self):
        return torch.softmax(torch.relu(self.nodevec1 @ self.nodevec2), dim=1)

    def forward(self, x_dem, x_ctx):               # x_dem [B,T,N], x_ctx [B,T,F]
        B, T, N = x_dem.shape
        e = self.emb.weight[None, None].expand(B, T, N, -1)
        c = x_ctx[:, :, None, :].expand(B, T, N, -1)
        x = torch.cat([x_dem[..., None], c, e], dim=-1).permute(0, 3, 2, 1)   # [B,D,N,T]
        if T < self.receptive_field:
            x = F.pad(x, (self.receptive_field - T, 0))
        supports = [getattr(self, f"S{i}") for i in range(self.n_fixed)]
        if self.adaptive:
            supports = supports + [self.adaptive_adj()]

        x = self.start(x)
        skip = None
        for i in range(len(self.filt)):
            res = x
            h = torch.tanh(self.filt[i](x)) * torch.sigmoid(self.gate[i](x))
            s = self.skip[i](h)
            skip = s if skip is None else s + skip[..., -s.size(3):]
            h = self.gconv[i](h, supports)
            x = self.bn[i](h + res[..., -h.size(3):])
        out = self.end2(torch.relu(self.end1(torch.relu(skip))))   # [B,H,N,1]
        return out[..., -1]                                        # [B,H,N]

# ----------GCN-GRU on each graph + random control ----------
for gname, A in graph_set.items():
    torch.manual_seed(42); np.random.seed(42)
    m, _ = fit_and_evaluate(GCNGRU(N, N_CTX, sym_norm(A)), f"GCN-GRU ({gname} graph)")
    del m; torch.cuda.empty_cache(); gc.collect()

# ---------- Select the best REAL graph on VALIDATION MAE (no test leakage) ----------
rv = results[(results["split"] == "val") & (results["subset"] == "all") &
             results["model"].str.startswith("GCN-GRU") &
             ~results["model"].str.contains(r"random|\[", regex=True)]
val_rank = rv.groupby("model")["MAE"].mean().sort_values()
print("\nValidation MAE (mean over horizons) by graph:")
print(val_rank.round(4))
best_model_name = val_rank.index[0]
best_gname = best_model_name.split("(")[1].replace(" graph)", "")
print(f"\nBest graph on validation: {best_gname}")

# ---------- Graph WaveNet ----------
torch.manual_seed(42); np.random.seed(42)
gwn_adp, _ = fit_and_evaluate(
    GraphWaveNet(N, N_CTX, supports=[], adaptive=True), "Graph WaveNet (adaptive only)")

torch.manual_seed(42); np.random.seed(42)
gwn_name = f"Graph WaveNet ({best_gname} + adaptive)"
gwn_best, _ = fit_and_evaluate(
    GraphWaveNet(N, N_CTX, supports=[row_norm(graph_set[best_gname])], adaptive=True), gwn_name)

# ---------- Graph ablation: which graph helps, relative to the no-graph GRU ----------
abl_models = (["GRU (no graph)"] + [f"GCN-GRU ({g} graph)" for g in graph_set] +
              ["Graph WaveNet (adaptive only)", gwn_name])
rt = (results[(results["split"] == "test") & (results["subset"] == "all") &
              results["model"].isin(abl_models)]
      .pivot_table(index="model", columns="horizon", values="MAE").reindex(abl_models))
for h in EVAL_H:
    rt[f"vs GRU h={h} (%)"] = (rt[h] / rt.loc["GRU (no graph)", h] - 1) * 100
print("\nGRAPH ABLATION — TEST MAE (all zones); negative % = better than no-graph GRU")
display(rt.round(3))

# ---------- Full leaderboard ----------
print("\nLEADERBOARD — TEST SET, ALL ZONES")
display(show("test", "all"))
print("\nLEADERBOARD — TEST SET, ACTIVE ZONES")
display(show("test", "active"))
print("\nLEADERBOARD — TEST SET, EVENT CELLS")
display(show("test", "events"))

# ---------- Whole-dataset view of the key models ----------
key_models = ["GRU (no graph)", "LightGBM + spatial lags", best_model_name, gwn_name]
print("\nMAE BY SPLIT — KEY MODELS (all zones)")
display(split_summary(key_models, "all", "MAE"))
generalization_gap(key_models)
plot_full_year(key_models, h=1)
print("Monthly MAE (1-hour-ahead, all zones):")
display(plot_monthly_mae(key_models, h=1))

# ---------- What did the adaptive graph learn? ----------
with torch.no_grad():
    adp = gwn_adp.adaptive_adj().cpu().numpy()
np.fill_diagonal(adp, 0)
top5 = np.argsort(-adp, axis=1)[:, :5]
r5 = np.repeat(np.arange(N), 5)
for gname in ["contiguity", "OD flow", "correlation"]:
    overlap = (graph_set[gname][r5, top5.ravel()] > 0).mean()
    print(f"Share of learned top-5 neighbours that are also {gname} neighbours: {overlap:.1%}")

print("\nStrongest learned connections:")
flat = np.argsort(-adp, axis=None)[:10]
for a, b in zip(*np.unravel_index(flat, adp.shape)):
    print(f"  {zone_info.iloc[a]['Zone']} → {zone_info.iloc[b]['Zone']}: "
          f"weight {adp[a, b]:.3f} ({D[a, b]:.1f} mi)")

print("\nGNN MODELLING COMPLETED!")

In [ ]:
from scipy import stats
import geopandas as gpd, glob, folium, branca.colormap as bcm

print("FINAL MODEL COMPARISON AND ERROR ANALYSIS")
print("=" * 60)

zone_list = final_demand_matrix.columns.tolist()
zone_info = zones.set_index("LocationID").loc[zone_list, ["Borough", "Zone"]]
borough = zone_info["Borough"].to_numpy()
n_test = len(test_o)
how = (idx.dayofweek * 24 + idx.hour).to_numpy()

# ---------- Select comparison models using VALIDATION results only ----------
val = results[(results["split"] == "val") & (results["subset"] == "all") & ~results["model"].str.contains(r"\[", regex=True)].groupby("model")["MAE"].mean()
baseline_names = [m for m in val.index
                  if m.startswith(("Persistence", "Seasonal", "Historical", "Weekly"))]
gnn_names = [m for m in val.index
             if (m.startswith("GCN-GRU") and "random" not in m) or m.startswith("Graph WaveNet")]
best_baseline = val[baseline_names].idxmin()
best_gnn = val[gnn_names].idxmin()

models = {
    "Best baseline": best_baseline,
    "LightGBM": "LightGBM (no spatial features)",
    "LightGBM + spatial": "LightGBM + spatial lags",
    "GRU (no graph)": "GRU (no graph)",
    "GCN-GRU random graph": "GCN-GRU (random (control) graph)",
    "Best GNN": best_gnn,
}
print(f"Best baseline (validation): {best_baseline}")
print(f"Best GNN (validation):      {best_gnn}")

# ---------- Prediction loaders ----------
ha_table = np.stack([counts[:train_end][how[:train_end] == k].mean(0) for k in range(168)])

def baseline_pred(name, orig, h):
    tgt = orig + h
    if name.startswith("Persistence"):   return counts[orig]
    if "yesterday" in name:              return counts[tgt - 24]
    if "last week" in name:              return counts[tgt - 168]
    if name.startswith("Historical"):    return ha_table[how[tgt]]
    if name.startswith("Weekly moving"): return np.mean([counts[tgt - 168 * k] for k in range(1, 5)], 0)

def load_full(name):
    if name in full_preds:
        return full_preds[name]
    if name.startswith("LightGBM"):
        path = SAVE + f"preds_lgbm_{'spatial' if 'spatial lags' in name else 'base'}_full.npz"
    else:
        path = SAVE + f"preds_{re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')}_full.npz"
    z = np.load(path)
    full_preds[name] = {h: z[f"h{h}"] for h in EVAL_H}
    return full_preds[name]

def test_pred(name, h):
    if name in baseline_names:
        return baseline_pred(name, test_o, h).astype(float)
    return np.clip(load_full(name)[h][-n_test:], 0, None)

def abs_err(name, h):
    tgt = test_o + h
    E = np.abs(test_pred(name, h) - counts[tgt]).astype(float)
    E[eval_exclude[tgt]] = np.nan          # imputed / DST cells excluded
    return E                                # [samples, zones]

# =====================================================================
# HEADLINE COMPARISON (TEST SET)
# =====================================================================
inv = {v: k for k, v in models.items()}
head = results[(results["split"] == "test") & results["model"].isin(models.values())]
for subset in ["all", "active"]:
    t = (head[head["subset"] == subset]
         .pivot_table(index="model", columns="horizon", values=["MAE", "RMSE", "WMAPE"])
         .rename(index=inv).reindex(list(models)))
    print(f"\n1. TEST RESULTS — {subset.upper()} ZONES")
    display(t.round(3))

r = head[head["subset"] == "all"].set_index(["model", "horizon"])["MAE"]
print("\nBest GNN MAE change vs each model (test, all zones; negative = GNN better):")
for label, name in models.items():
    if name != best_gnn:
        print(f"  vs {label:22s} " + " | ".join(
            f"h={h}: {(r[(best_gnn, h)] / r[(name, h)] - 1) * 100:+.1f}%" for h in EVAL_H))

# =====================================================================
# STATISTICAL SIGNIFICANCE — DIEBOLD–MARIANO TEST
# =====================================================================
def dm_test(e_a, e_b, lag=24):
    """DM test on hourly MAE differences with Newey–West variance (24-hour lag window)."""
    d = np.nanmean(e_a, 1) - np.nanmean(e_b, 1)
    d = d[~np.isnan(d)]
    n, dbar = len(d), d.mean()
    if n < 3:
        return np.nan, np.nan, dbar
    lag = min(lag, n - 1)
    dc = d - dbar
    lrv = dc @ dc / n + 2 * sum((1 - k / (lag + 1)) * (dc[k:] @ dc[:-k]) / n
                                for k in range(1, lag + 1))
    if lrv <= 0:
        return (0.0, 1.0, dbar) if dbar == 0 else (np.nan, np.nan, dbar)
    stat = dbar / np.sqrt(lrv / n)
    return stat, 2 * (1 - stats.norm.cdf(abs(stat))), dbar

dm_rows = []
for h in EVAL_H:
    e_gnn = abs_err(best_gnn, h)
    for label, name in models.items():
        if name == best_gnn:
            continue
        stat, p, diff = dm_test(e_gnn, abs_err(name, h))
        dm_rows.append({"horizon": h, "comparison": f"Best GNN vs {label}",
                        "mean_MAE_difference": diff, "DM_statistic": stat,
                        "p_value": p, "significant_5%": p < 0.05})
dm_table = pd.DataFrame(dm_rows)
print("\n2. DIEBOLD–MARIANO TEST (negative statistic = Best GNN has lower error)")
display(dm_table.round(4))

# =====================================================================
#  WHERE DO ERRORS HAPPEN? (1-hour-ahead, test set)
# =====================================================================
H_A = 1
tgt = test_o + H_A
E = {label: abs_err(name, H_A) for label, name in models.items()}
Y = counts[tgt].astype(float); Y[eval_exclude[tgt]] = np.nan

# By borough
bor = ["Manhattan", "Queens", "Brooklyn", "Bronx", "Staten Island", "EWR"]
bor_tab = pd.DataFrame({lab: {b: np.nanmean(E[lab][:, borough == b]) for b in bor} for lab in E})
bor_tab["Best GNN vs GRU (%)"] = (bor_tab["Best GNN"] / bor_tab["GRU (no graph)"] - 1) * 100
print("\n3a. MAE BY BOROUGH (1-hour-ahead)")
display(bor_tab.round(3))

# By zone demand level (WMAPE, so busy and quiet zones are comparable)
lvl = pd.cut(counts[:train_end].mean(0), [-0.01, 1, 10, 50, np.inf],
             labels=["<1/hr", "1–10/hr", "10–50/hr", "50+/hr"])
lvl_tab = pd.DataFrame({lab: {str(c): np.nansum(E[lab][:, lvl == c]) / np.nansum(Y[:, lvl == c])
                              for c in lvl.categories} for lab in E})
lvl_tab.insert(0, "zones", [int((lvl == c).sum()) for c in lvl.categories])
print("\n3b. WMAPE BY ZONE DEMAND LEVEL (training mean pickups/hour)")
display(lvl_tab.round(3))

# By condition
wd = idx[tgt].dayofweek < 5
hol = context["is_holiday"].to_numpy()[tgt] == 1
ev = event_flags[tgt]
cond_tab = pd.DataFrame({lab: {
    "Weekday hours": np.nanmean(E[lab][wd]), "Weekend hours": np.nanmean(E[lab][~wd]),
    "Holiday hours": np.nanmean(E[lab][hol]), "Non-holiday hours": np.nanmean(E[lab][~hol]),
    "Event cells": np.nanmean(E[lab][ev]), "Normal cells": np.nanmean(E[lab][~ev]),
} for lab in E})
print("\n3c. MAE BY CONDITION (1-hour-ahead)")
display(cond_tab.round(3))

# By hour of day and by day across the test period
hrs = idx[tgt].hour.to_numpy()
hour_tab = pd.DataFrame({lab: pd.Series(np.nanmean(E[lab], 1)).groupby(hrs).mean() for lab in E})
daily_tab = pd.DataFrame({lab: pd.Series(np.nanmean(E[lab], 1), index=idx[tgt]).resample("D").mean()
                          for lab in E})
fig, ax = plt.subplots(2, 1, figsize=(14, 8))
hour_tab.plot(ax=ax[0], marker="o", title="MAE by Hour of Day (Test, 1-Hour-Ahead)")
ax[0].set_xlabel("Hour of day"); ax[0].set_ylabel("MAE (pickups)"); ax[0].set_xticks(range(24))
daily_tab.plot(ax=ax[1], title="Daily MAE Across the Test Period (Nov–Dec 2025)")
for d in ["2025-11-27", "2025-12-25"]:
    ax[1].axvline(pd.Timestamp(d), ls="--", c="gray")
ax[1].set_ylabel("MAE (pickups)")
for a in ax: a.legend(fontsize=8)
plt.tight_layout(); plt.show()

# =====================================================================
# WHERE DOES THE GRAPH HELP? (per-zone improvement map)
# =====================================================================
act = counts[:train_end].mean(0) >= 1
zone_gru = np.nanmean(E["GRU (no graph)"], 0)
zone_gnn = np.nanmean(E["Best GNN"], 0)
zone_pct = np.where(act, (zone_gnn / zone_gru - 1) * 100, np.nan)

print("\n4. PER-ZONE IMPROVEMENT (Best GNN vs GRU, active zones)")
print(f"Zones improved: {(zone_pct < 0).sum()} of {act.sum()} active zones")
print(f"Median change: {np.nanmedian(zone_pct):+.1f}%")
deg = (G["A_adj"] > 0).sum(1)
for lab, x in [("training mean demand", counts[:train_end].mean(0)), ("contiguity degree", deg)]:
    rho, p = stats.spearmanr(x[act], zone_pct[act])
    print(f"Spearman correlation of improvement with {lab}: {rho:+.3f} (p = {p:.3f})")

if "gdf" not in globals():
    !wget -q -nc https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip && unzip -oq taxi_zones.zip -d tz
    gdf = gpd.read_file(glob.glob("tz/**/*.shp", recursive=True)[0]).to_crs(2263)
    gdf = gdf.dissolve(by="LocationID").reindex(zone_list)

imp = gdf[["geometry"]].reset_index(drop=True).copy()
imp["Zone"], imp["Borough"] = zone_info["Zone"].values, borough
imp["GRU_MAE"], imp["GNN_MAE"] = zone_gru.round(2), zone_gnn.round(2)
imp["change_pct"] = np.round(zone_pct, 1)

# Static map for the paper
ax = imp.plot(column=imp["change_pct"].clip(-30, 30), cmap="RdBu_r", vmin=-30, vmax=30,
              legend=True, figsize=(10, 10), missing_kwds={"color": "lightgray"},
              legend_kwds={"label": "MAE change vs GRU (%) — blue = GNN better", "shrink": 0.6})
ax.set_title(f"Where the Graph Helps: {best_gnn} vs GRU (Test, 1-Hour-Ahead)"); ax.axis("off")
plt.show()

# Interactive map
imp_ll = imp.to_crs(4326)
imp_ll["change_pct"] = imp_ll["change_pct"].astype(float)
cmap = bcm.LinearColormap(["#2166ac", "#f7f7f7", "#b2182b"], vmin=-30, vmax=30,
                          caption="MAE change vs GRU (%) — blue = GNN better")
m = folium.Map(location=[40.73, -73.94], zoom_start=11, tiles="cartodbpositron")
folium.GeoJson(
    imp_ll,
    style_function=lambda f: {
        "fillColor": "#d9d9d9" if f["properties"]["change_pct"] is None or
                     np.isnan(f["properties"]["change_pct"])
                     else cmap(np.clip(f["properties"]["change_pct"], -30, 30)),
        "color": "#555", "weight": 0.4, "fillOpacity": 0.7},
    highlight_function=lambda f: {"weight": 2, "color": "black"},
    tooltip=folium.GeoJsonTooltip(
        fields=["Zone", "Borough", "GRU_MAE", "GNN_MAE", "change_pct"],
        aliases=["Zone", "Borough", "GRU MAE", "Best GNN MAE", "Change (%)"]),
).add_to(m)
cmap.add_to(m)
m.save(SAVE + "gnn_improvement_map.html")
display(m)

# =====================================================================
# EXAMPLE FORECASTS — THANKSGIVING WEEK
# =====================================================================
wk = (idx[tgt] >= "2025-11-24") & (idx[tgt] < "2025-12-01")
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, z in zip(axes, [161, 132, 79]):
    j = zone_list.index(z)
    ax.plot(idx[tgt][wk], counts[tgt][wk, j], color="black", lw=1.8, label="Actual")
    for lab in ["Best GNN", "GRU (no graph)", "LightGBM + spatial"]:
        ax.plot(idx[tgt][wk], test_pred(models[lab], H_A)[wk, j], lw=1, alpha=0.85, label=lab)
    ax.axvspan(pd.Timestamp("2025-11-27"), pd.Timestamp("2025-11-28"), color="gray", alpha=0.15)
    ax.set_title(f"{zone_info.loc[z, 'Zone']} — 1-Hour-Ahead Forecasts, Thanksgiving Week (shaded)")
    ax.set_ylabel("Pickups")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

# =====================================================================
# EXPORT FINAL TABLES
# =====================================================================
final_table = (results[(results["split"] == "test") & results["subset"].isin(["all", "active"])]
               .pivot_table(index=["subset", "model"], columns="horizon",
                            values=["MAE", "RMSE", "WMAPE"]).round(3))
final_table.to_csv(SAVE + "final_test_results.csv")
dm_table.to_csv(SAVE + "diebold_mariano_tests.csv", index=False)
bor_tab.to_csv(SAVE + "mae_by_borough.csv")
lvl_tab.to_csv(SAVE + "wmape_by_demand_level.csv")
cond_tab.to_csv(SAVE + "mae_by_condition.csv")
print("\nTables saved to Drive: final_test_results, diebold_mariano_tests, mae_by_borough, "
      "wmape_by_demand_level, mae_by_condition")

print("\nFINAL ANALYSIS COMPLETED!")

In [ ]:
print(" ROBUSTNESS — RANDOM SEEDS AND CONTEXT-FEATURE ABLATION")
print("=" * 65)

# ---------- Helpers ----------
if "best_gnn" not in globals():
    val = results[(results["split"] == "val") & (results["subset"] == "all") & ~results["model"].str.contains(r"\[", regex=True)].groupby("model")["MAE"].mean()
    best_gnn = val[[m for m in val.index if (m.startswith("GCN-GRU") and "random" not in m)
                    or m.startswith("Graph WaveNet")]].idxmin()
print("Best GNN (chosen on validation in Step 25):", best_gnn)

def build_model(name):
    """Rebuild an untrained model from its name (same architecture as Steps 23–24)."""
    if name == "GRU (no graph)":
        return NodeGRU(N, N_CTX)
    if name.startswith("GCN-GRU"):
        g = name.split("(", 1)[1].rsplit(" graph)", 1)[0]
        return GCNGRU(N, N_CTX, sym_norm(graph_set[g]))
    if name == "Graph WaveNet (adaptive only)":
        return GraphWaveNet(N, N_CTX, supports=[], adaptive=True)
    if name.startswith("Graph WaveNet"):
        g = name.split("(", 1)[1].rsplit(" + adaptive)", 1)[0]
        return GraphWaveNet(N, N_CTX, supports=[row_norm(graph_set[g])], adaptive=True)
    raise ValueError(name)

class ContextMask(nn.Module):
    """Wraps a model and zeroes the context features that are NOT kept.
    Zero = training mean for standardized weather, and 'no holiday' for flags."""
    def __init__(self, base, keep_cols):
        super().__init__()
        self.base = base
        mask = torch.zeros(N_CTX)
        mask[[ctx_cols.index(c) for c in keep_cols]] = 1
        self.register_buffer("mask", mask)
    def forward(self, x_dem, x_ctx):
        return self.base(x_dem, x_ctx * self.mask)

def get_full(name):
    if name not in full_preds:
        z = np.load(SAVE + f"preds_{re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')}_full.npz")
        full_preds[name] = {h: z[f"h{h}"] for h in EVAL_H}
    return full_preds[name]

def test_abs_err(name, h):
    tgt = test_o + h
    P = np.clip(get_full(name)[h][-len(test_o):], 0, None)
    E = np.abs(P - counts[tgt]).astype(float)
    E[eval_exclude[tgt]] = np.nan
    return E

def test_mae(name):
    r = results[(results["model"] == name) & (results["split"] == "test") & (results["subset"] == "all")]
    return r.set_index("horizon")["MAE"]

# =====================================================================
# RANDOM-SEED ROBUSTNESS (seed 42 already trained in Steps 23–24)
# =====================================================================
EXTRA_SEEDS = [1, 2]
seed_models = ["GRU (no graph)", best_gnn]

for base in seed_models:
    for s in EXTRA_SEEDS:
        torch.manual_seed(s); np.random.seed(s)
        m, _ = fit_and_evaluate(build_model(base), f"{base} [seed {s}]")
        del m; torch.cuda.empty_cache(); gc.collect()

rows = []
for base in seed_models:
    runs = [base] + [f"{base} [seed {s}]" for s in EXTRA_SEEDS]
    maes = pd.DataFrame({r: test_mae(r) for r in runs})          # horizons × runs
    for h in EVAL_H:
        rows.append({"model": base, "horizon": h,
                     "MAE_mean": maes.loc[h].mean(), "MAE_std": maes.loc[h].std(),
                     "MAE_min": maes.loc[h].min(), "MAE_max": maes.loc[h].max()})
seed_table = pd.DataFrame(rows)
print("\nA. TEST MAE ACROSS 3 RANDOM SEEDS (42, 1, 2) — ALL ZONES")
display(seed_table.round(4))

print("\nIs the GNN better than the GRU in every seed pairing?")
for h in EVAL_H:
    gru_runs = [test_mae(r)[h] for r in ["GRU (no graph)"] + [f"GRU (no graph) [seed {s}]" for s in EXTRA_SEEDS]]
    gnn_runs = [test_mae(r)[h] for r in [best_gnn] + [f"{best_gnn} [seed {s}]" for s in EXTRA_SEEDS]]
    wins = sum(g < r for g, r in zip(gnn_runs, gru_runs))
    gap = (np.mean(gnn_runs) / np.mean(gru_runs) - 1) * 100
    print(f"  h={h}: GNN better in {wins}/3 seeds | mean change {gap:+.1f}% | "
          f"worst GNN run {max(gnn_runs):.3f} vs best GRU run {min(gru_runs):.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
for k, base in enumerate(seed_models):
    st = seed_table[seed_table["model"] == base]
    ax.errorbar(np.array(EVAL_H) + (k - 0.5) * 0.15, st["MAE_mean"], yerr=st["MAE_std"],
                marker="o", capsize=4, label=base)
ax.set_xticks(EVAL_H); ax.set_xlabel("Forecast horizon (hours)"); ax.set_ylabel("Test MAE (pickups)")
ax.set_title("Test MAE Mean ± Std Across 3 Random Seeds"); ax.legend(); plt.show()

# =====================================================================
# CONTEXT-FEATURE ABLATION ON THE BEST GNN (seed 42)
# =====================================================================
holiday_cols = ["is_holiday", "pre_holiday", "post_holiday"]
ablations = {
    "no weather": [c for c in ctx_cols if c not in wx_cols],
    "no holiday flags": [c for c in ctx_cols if c not in holiday_cols],
    "no context features": [],
}
for label, keep in ablations.items():
    torch.manual_seed(42); np.random.seed(42)
    m, _ = fit_and_evaluate(ContextMask(build_model(best_gnn), keep), f"{best_gnn} [{label}]")
    del m; torch.cuda.empty_cache(); gc.collect()

abl_names = {"full model": best_gnn, **{k: f"{best_gnn} [{k}]" for k in ablations}}
abl = pd.DataFrame({k: test_mae(v) for k, v in abl_names.items()}).T
for h in EVAL_H:
    abl[f"change h={h} (%)"] = (abl[h] / abl.loc["full model", h] - 1) * 100
print("\nB. CONTEXT ABLATION — TEST MAE, ALL ZONES (positive % = worse without the feature)")
display(abl.round(3))

# Where context should matter most: rainy hours and holiday hours (1-hour-ahead)
tgt = test_o + 1
rain = context["is_raining"].to_numpy()[tgt] == 1
hol = context["is_holiday"].to_numpy()[tgt] == 1
cond = pd.DataFrame({k: {"Rainy hours": np.nanmean(test_abs_err(v, 1)[rain]),
                         "Dry hours": np.nanmean(test_abs_err(v, 1)[~rain]),
                         "Holiday hours": np.nanmean(test_abs_err(v, 1)[hol]),
                         "Non-holiday hours": np.nanmean(test_abs_err(v, 1)[~hol])}
                     for k, v in abl_names.items()})
print(f"\nMAE BY CONDITION (1-hour-ahead) — rainy test hours: {rain.sum()}, holiday test hours: {hol.sum()}")
display(cond.round(3))

abl[[f"change h={h} (%)" for h in EVAL_H]].drop("full model").plot(
    kind="bar", figsize=(9, 4), title="Effect of Removing Context Features on Test MAE")
plt.ylabel("MAE change vs full model (%)"); plt.axhline(0, c="black", lw=0.8)
plt.xticks(rotation=0); plt.show()

seed_table.to_csv(SAVE + "seed_robustness.csv", index=False)
abl.to_csv(SAVE + "context_ablation.csv")
cond.to_csv(SAVE + "context_ablation_by_condition.csv")
print("\nROBUSTNESS CHECKS COMPLETED!")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import folium, branca.colormap as bcm

print("FORECASTING SYSTEM DEMO")
print("=" * 55)

slug = lambda n: re.sub(r"[^a-z0-9]+", "_", n.lower()).strip("_")
zone_list = final_demand_matrix.columns.tolist()
zone_info = zones.set_index("LocationID").loc[zone_list, ["Borough", "Zone"]]
how = (idx.dayofweek * 24 + idx.hour).to_numpy()
if "ha_table" not in globals():
    ha_table = np.stack([counts[:train_end][how[:train_end] == k].mean(0) for k in range(168)])

# ---------- Load the trained best GNN ----------
system_model = build_model(best_gnn).to(device)
system_model.load_state_dict(torch.load(SAVE + f"model_{slug(best_gnn)}.pt", map_location=device))
system_model.eval()
print("Forecasting model:", best_gnn)

base_geo = gdf[["geometry"]].reset_index(drop=True).to_crs(4326)

# ---------- Core forecasting function ----------
@torch.no_grad()
def forecast_at(ts):
    """Forecast pickups for all zones for the next H hours, using the last T_IN hours."""
    ts = pd.Timestamp(ts).floor("h")
    o = idx.get_loc(ts)
    if o < T_IN - 1:
        raise ValueError(f"Need at least {T_IN} hours of history before the forecast time.")
    t0 = time.time()
    win = o + win_off
    logp = system_model(Ls_t[win][None], C_t[win][None])[0] * sd_t + mu_t     # [H, N]
    pred = torch.expm1(logp).clamp_min(0).cpu().numpy()
    latency_ms = (time.time() - t0) * 1000

    df = pd.DataFrame({"LocationID": zone_list,
                       "Zone": zone_info["Zone"].values,
                       "Borough": zone_info["Borough"].values,
                       "now": counts[o]})
    for h in range(1, H + 1):
        df[f"+{h}h"] = pred[h - 1].round(1)
        df[f"actual_+{h}h"] = counts[o + h] if o + h < T else np.nan
        df[f"typical_+{h}h"] = ha_table[how[o + h]].round(1) if o + h < T else np.nan
    return ts, o, df, latency_ms

# ---------- Map builder ----------
def forecast_map(ts, df, horizons=(1, 3, 6)):
    m = folium.Map(location=[40.73, -73.94], zoom_start=11, tiles=None)
    folium.TileLayer("cartodbpositron", control=False).add_to(m)
    vmax = np.log1p(df[[f"+{h}h" for h in horizons]].to_numpy().max())
    cmap = bcm.linear.YlOrRd_09.scale(0, vmax)
    cmap.caption = "log(1 + forecast pickups)"
    for k, h in enumerate(horizons):
        g = base_geo.copy()
        g["Zone"], g["Borough"] = df["Zone"].values, df["Borough"].values
        g["now"] = df["now"].astype(int).values
        g["forecast"] = df[f"+{h}h"].values
        g["actual"] = [f"{v:.0f}" if np.isfinite(v) else "n/a" for v in df[f"actual_+{h}h"]]
        g["typical"] = [f"{v:.1f}" if np.isfinite(v) else "n/a" for v in df[f"typical_+{h}h"]]
        when = (ts + pd.Timedelta(hours=h)).strftime("%a %b %d, %H:%M")
        fg = folium.FeatureGroup(name=f"Forecast +{h}h ({when})", overlay=False, show=(k == 0))
        folium.GeoJson(
            g,
            style_function=lambda f: {"fillColor": cmap(np.log1p(f["properties"]["forecast"])),
                                      "color": "#555", "weight": 0.3, "fillOpacity": 0.75},
            highlight_function=lambda f: {"weight": 2, "color": "black"},
            tooltip=folium.GeoJsonTooltip(
                fields=["Zone", "Borough", "now", "forecast", "actual", "typical"],
                aliases=["Zone", "Borough", "Pickups now", f"Forecast (+{h}h)",
                         "Actual", "Typical for this hour"]),
        ).add_to(fg)
        fg.add_to(m)
    cmap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

# ---------- Full report for one forecast time ----------
def forecast_report(ts, zone_id=161, show_map=True):
    ts, o, df, latency = forecast_at(ts)
    split = "TRAIN (in-sample)" if ts < TRAIN_END else ("VALIDATION" if ts < VAL_END else "TEST (out-of-sample)")
    print(f"Forecast issued at {ts:%A %Y-%m-%d %H:%M} | period: {split} | "
          f"inference time: {latency:.1f} ms for {N} zones × {H} hours")

    # Citywide summary
    city = pd.DataFrame({
        "target time": [(ts + pd.Timedelta(hours=h)).strftime("%a %H:%M") for h in range(1, H + 1)],
        "forecast": [df[f"+{h}h"].sum() for h in range(1, H + 1)],
        "actual": [df[f"actual_+{h}h"].sum() if df[f"actual_+{h}h"].notna().all() else np.nan
                   for h in range(1, H + 1)],
    }, index=[f"+{h}h" for h in range(1, H + 1)])
    city["error_%"] = (city["forecast"] / city["actual"] - 1) * 100
    print(f"\nCitywide pickups now: {int(df['now'].sum()):,}")
    display(city.round(1))

    # Hotspots
    print("\nTop 10 zones by forecast demand in the next hour:")
    display(df.nlargest(10, "+1h")[["Zone", "Borough", "now", "+1h", "actual_+1h", "+3h", "+6h"]])
    rising = df[df["+3h"] >= 20].assign(change=lambda d: d["+3h"] - d["now"])
    print("\nZones with the largest predicted INCREASE over the next 3 hours (forecast ≥ 20):")
    display(rising.nlargest(10, "change")[["Zone", "Borough", "now", "+3h", "actual_+3h", "change"]])

    # Zone view: last 24 hours + 6-hour forecast
    j = zone_list.index(zone_id)
    past = np.arange(o - 23, o + 1)
    fut = np.arange(o + 1, min(o + H, T - 1) + 1)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(idx[past], counts[past, j], color="black", marker=".", label="Observed (last 24 h)")
    ax.plot([idx[o]] + [ts + pd.Timedelta(hours=h) for h in range(1, H + 1)],
            [counts[o, j]] + [df.loc[j, f"+{h}h"] for h in range(1, H + 1)],
            color="tab:red", marker="o", label=f"Forecast ({best_gnn})")
    if len(fut):
        ax.plot(idx[fut], counts[fut, j], color="black", ls="--", marker="x", label="Actual (future)")
        ax.plot(idx[fut], ha_table[how[fut], j], color="gray", ls=":", label="Typical for this hour")
    ax.axvline(ts, color="gray", lw=0.8)
    ax.set_title(f"{zone_info.loc[zone_id, 'Zone']} ({zone_info.loc[zone_id, 'Borough']}) — "
                 f"forecast issued {ts:%a %b %d, %H:%M}")
    ax.set_ylabel("Pickups per hour"); ax.legend(fontsize=8); plt.show()

    df.to_csv(SAVE + f"forecast_{ts:%Y%m%d_%H%M}.csv", index=False)
    if show_map:
        m = forecast_map(ts, df)
        m.save(SAVE + f"forecast_map_{ts:%Y%m%d_%H%M}.html")
        display(m)
    return df

# ---------- Demo: the evening before Thanksgiving (out-of-sample) ----------
demo_df = forecast_report("2025-11-26 16:00", zone_id=161)

# ---------- Interactive control panel ----------
date_w = widgets.DatePicker(value=pd.Timestamp("2025-12-05").date(), description="Date")
hour_w = widgets.IntSlider(value=17, min=0, max=23, description="Hour")
zone_w = widgets.Dropdown(
    options=sorted([(f"{zone_info.loc[z, 'Zone']} ({zone_info.loc[z, 'Borough']})", z)
                    for z in zone_list]),
    value=161, description="Zone", layout=widgets.Layout(width="450px"))
map_w = widgets.Checkbox(value=True, description="Show map")
run_w = widgets.Button(description="Run forecast", button_style="primary")
out_w = widgets.Output()

def on_run(_):
    with out_w:
        clear_output(wait=True)
        if date_w.value is None:
            print("Choose a date."); return
        ts = pd.Timestamp(date_w.value) + pd.Timedelta(hours=hour_w.value)
        if ts < idx[T_IN - 1] or ts > idx[-1]:
            print("Choose a time within 2025 (after Jan 1, 11:00)."); return
        forecast_report(ts, zone_id=zone_w.value, show_map=map_w.value)

run_w.on_click(on_run)
print("\nINTERACTIVE FORECASTING PANEL — choose a time and zone, then click 'Run forecast'")
display(widgets.VBox([widgets.HBox([date_w, hour_w]), widgets.HBox([zone_w, map_w]), run_w, out_w]))

In [ ]:
import sys, platform, os

print("PROJECT SUMMARY, REPRODUCIBILITY, AND PAPER-READY OUTPUTS")
print("=" * 65)

# =====================================================================
# REPRODUCIBILITY RECORD
# =====================================================================
import sklearn, scipy, statsmodels, lightgbm, geopandas, folium, holidays, pyarrow
versions = {
    "Python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
    "torch": torch.__version__, "lightgbm": lightgbm.__version__, "scipy": scipy.__version__,
    "statsmodels": statsmodels.__version__, "geopandas": geopandas.__version__,
    "folium": folium.__version__, "holidays": holidays.__version__, "pyarrow": pyarrow.__version__,
    "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
}
print("1. ENVIRONMENT")
for k, v in versions.items():
    print(f"  {k:12s} {v}")
!pip freeze | grep -iE "^(pandas|numpy|torch|lightgbm|geopandas|folium|scipy|statsmodels|holidays|matplotlib|pyarrow|ipywidgets|branca|mapclassify)==" > "{SAVE}requirements.txt"
print(f"  requirements.txt saved to Drive")

# =====================================================================
# ARTIFACT INVENTORY
# =====================================================================
def category(f):
    if f.startswith("demand_") or f in ("dst_flag.parquet", "weather_2025.parquet",
                                         "context_features.parquet", "prep.npz"):
        return "Data"
    if f.startswith(("graphs", "A_adj", "D_miles")): return "Graphs"
    if f.startswith(("model_", "lgbm_")) and not f.endswith(".csv"): return "Trained models"
    if f.startswith("preds_"): return "Predictions"
    if f.endswith(".html"): return "Interactive maps"
    if f.endswith((".csv", ".tex", ".md")): return "Results tables"
    return "Other"

files = [f for f in sorted(os.listdir(SAVE)) if os.path.isfile(SAVE + f)]
inv = pd.DataFrame({"file": files,
                    "size_MB": [os.path.getsize(SAVE + f) / 1e6 for f in files]})
inv["category"] = inv["file"].map(category)
print("\n2. SAVED ARTIFACTS")
display(inv.groupby("category").agg(files=("file", "count"), total_MB=("size_MB", "sum")).round(1))

# =====================================================================
# KEY FINDINGS (auto-generated from saved results)
# =====================================================================
results = pd.read_csv(SAVE + "model_results.csv")
main = results[~results["model"].str.contains(r"\[", regex=True)]      # exclude seed/ablation runs
test_all = main[(main["split"] == "test") & (main["subset"] == "all")]
val_all = main[(main["split"] == "val") & (main["subset"] == "all")].groupby("model")["MAE"].mean()

baseline_names = [m for m in val_all.index if m.startswith(("Persistence", "Seasonal", "Historical", "Weekly"))]
gnn_names = [m for m in val_all.index if (m.startswith("GCN-GRU") and "random" not in m) or m.startswith("Graph WaveNet")]
best_baseline = val_all[baseline_names].idxmin()
best_gnn = val_all[gnn_names].idxmin()
mae = test_all.set_index(["model", "horizon"])["MAE"]

def pct(a, b, h):
    return (mae[(a, h)] / mae[(b, h)] - 1) * 100

print("\n3. KEY FINDINGS")
print(f"• Data: {int(counts.sum()):,} cleaned pickups, {T:,} hours × {N} zones (2025).")
print(f"• Best GNN (selected on validation): {best_gnn}")
print(f"• Best baseline (selected on validation): {best_baseline}")
for h in EVAL_H:
    print(f"• {h}-hour-ahead test MAE: Best GNN {mae[(best_gnn, h)]:.3f} | "
          f"vs best baseline {pct(best_gnn, best_baseline, h):+.1f}% | "
          f"vs LightGBM + spatial {pct(best_gnn, 'LightGBM + spatial lags', h):+.1f}% | "
          f"vs GRU (no graph) {pct(best_gnn, 'GRU (no graph)', h):+.1f}% | "
          f"vs random graph {pct(best_gnn, 'GCN-GRU (random (control) graph)', h):+.1f}%")

graph_models = [m for m in test_all["model"].unique() if m.startswith("GCN-GRU")]
best_graph_test = min(graph_models, key=lambda m: test_all[test_all["model"] == m]["MAE"].mean())
print(f"• Graph with lowest average test MAE (GCN-GRU family): {best_graph_test}")

if os.path.exists(SAVE + "diebold_mariano_tests.csv"):
    dm = pd.read_csv(SAVE + "diebold_mariano_tests.csv")
    sig = dm[(dm["significant_5%"]) & (dm["DM_statistic"] < 0)]
    print(f"• Diebold–Mariano: Best GNN significantly better in {len(sig)} of {len(dm)} comparisons (5% level).")
    nonsig = dm[~((dm["significant_5%"]) & (dm["DM_statistic"] < 0))]
    for _, row in nonsig.iterrows():
        print(f"    not significant / not better: {row['comparison']} (h={row['horizon']}, p={row['p_value']:.3f})")

if os.path.exists(SAVE + "seed_robustness.csv"):
    sr = pd.read_csv(SAVE + "seed_robustness.csv")
    for h in EVAL_H:
        g = sr[(sr["model"] == best_gnn) & (sr["horizon"] == h)].iloc[0]
        r_ = sr[(sr["model"] == "GRU (no graph)") & (sr["horizon"] == h)].iloc[0]
        print(f"• Seeds (h={h}): GNN {g['MAE_mean']:.3f} ± {g['MAE_std']:.3f} vs "
              f"GRU {r_['MAE_mean']:.3f} ± {r_['MAE_std']:.3f}")

if os.path.exists(SAVE + "context_ablation.csv"):
    ab = pd.read_csv(SAVE + "context_ablation.csv", index_col=0)
    for row in ab.index.drop("full model"):
        print(f"• Removing {row}: test MAE change " +
              ", ".join(f"h={h}: {ab.loc[row, f'change h={h} (%)']:+.1f}%" for h in EVAL_H))

# =====================================================================
# PAPER-READY RESULTS TABLE
# =====================================================================
paper_models = ([best_baseline, "LightGBM (no spatial features)", "LightGBM + spatial lags",
                 "GRU (no graph)"] + sorted(graph_models) +
                [m for m in test_all["model"].unique() if m.startswith("Graph WaveNet")])
paper_table = (test_all[test_all["model"].isin(paper_models)]
               .pivot_table(index="model", columns="horizon", values=["MAE", "RMSE", "WMAPE"])
               .reindex(paper_models))
paper_table.columns = [f"{m} (h={h})" for m, h in paper_table.columns]
paper_table = paper_table[[f"{m} (h={h})" for h in EVAL_H for m in ["MAE", "RMSE", "WMAPE"]]]
print("\n4. MAIN RESULTS TABLE (test set, all zones)")
display(paper_table.round(3))

paper_table.round(3).to_csv(SAVE + "paper_main_table.csv")
paper_table.to_latex(SAVE + "paper_main_table.tex", float_format="%.3f",
                     caption="Test-set forecasting accuracy (Nov–Dec 2025), all 263 zones.",
                     label="tab:main_results")
with open(SAVE + "paper_main_table.md", "w") as fh:
    fh.write(paper_table.round(3).to_markdown())
print("Saved: paper_main_table.csv / .tex / .md")

# =====================================================================
#PUBLICATION-QUALITY FIGURES (300 dpi PNG + PDF)
# =====================================================================
FIG = SAVE + "figures/"
os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"savefig.dpi": 300, "font.size": 11})

def save_fig(fig, name):
    fig.savefig(FIG + name + ".png", bbox_inches="tight")
    fig.savefig(FIG + name + ".pdf", bbox_inches="tight")
    plt.show()

short = {best_baseline: "Best baseline", "LightGBM (no spatial features)": "LightGBM",
         "LightGBM + spatial lags": "LightGBM + spatial", "GRU (no graph)": "GRU (no graph)"}
short.update({m: m.replace("GCN-GRU (", "GCN-GRU\n(").replace(" graph)", ")") for m in graph_models})
short.update({m: m.replace("Graph WaveNet (", "GWN\n(") for m in paper_models if m.startswith("Graph WaveNet")})

# demand over 2025 with data split
daily = final_demand_matrix.sum(axis=1).resample("D").sum()
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(daily, color="black", lw=1)
ax.axvspan(idx[0], TRAIN_END, alpha=0.12, color="tab:blue", label="Train")
ax.axvspan(TRAIN_END, VAL_END, alpha=0.15, color="tab:orange", label="Validation")
ax.axvspan(VAL_END, idx[-1], alpha=0.15, color="tab:green", label="Test")
ax.set_ylabel("Pickups per day"); ax.set_title("NYC Yellow Taxi Daily Pickups, 2025")
ax.legend(loc="lower left"); save_fig(fig, "fig1_daily_demand_split")

# test MAE by model and horizon
mt = test_all[test_all["model"].isin(paper_models)].pivot_table(
    index="model", columns="horizon", values="MAE").reindex(paper_models)
fig, ax = plt.subplots(figsize=(12, 4.5))
mt.rename(index=short).plot(kind="bar", ax=ax, width=0.8)
ax.set_ylabel("Test MAE (pickups per zone-hour)"); ax.set_xlabel("")
ax.set_title("Forecast Error by Model and Horizon (Test Set, All Zones)")
ax.legend(title="Horizon (h)"); plt.xticks(rotation=0, fontsize=8)
save_fig(fig, "fig2_mae_by_model")

# graph ablation relative to the no-graph GRU
abl_models = graph_models + [m for m in paper_models if m.startswith("Graph WaveNet")]
rel = pd.DataFrame({h: [pct(m, "GRU (no graph)", h) for m in abl_models] for h in EVAL_H},
                   index=[short[m] for m in abl_models])
fig, ax = plt.subplots(figsize=(11, 4))
rel.plot(kind="bar", ax=ax, width=0.8)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("MAE change vs GRU (%)"); ax.set_title("Effect of Graph Structure (negative = better than no graph)")
ax.legend(title="Horizon (h)"); plt.xticks(rotation=0, fontsize=8)
save_fig(fig, "fig3_graph_ablation")

# error growth with horizon
fig, ax = plt.subplots(figsize=(8, 4))
for m in [best_baseline, "LightGBM + spatial lags", "GRU (no graph)", best_gnn]:
    ax.plot(EVAL_H, [mae[(m, h)] for h in EVAL_H], marker="o", label=short.get(m, m).replace("\n", " "))
ax.set_xticks(EVAL_H); ax.set_xlabel("Forecast horizon (hours)"); ax.set_ylabel("Test MAE")
ax.set_title("Error Growth with Forecast Horizon"); ax.legend(fontsize=9)
save_fig(fig, "fig4_mae_vs_horizon")

# seed robustness
if os.path.exists(SAVE + "seed_robustness.csv"):
    fig, ax = plt.subplots(figsize=(8, 4))
    for k, m in enumerate(["GRU (no graph)", best_gnn]):
        s = sr[sr["model"] == m]
        ax.errorbar(np.array(EVAL_H) + (k - 0.5) * 0.15, s["MAE_mean"], yerr=s["MAE_std"],
                    marker="o", capsize=4, label=short.get(m, m).replace("\n", " "))
    ax.set_xticks(EVAL_H); ax.set_xlabel("Forecast horizon (hours)"); ax.set_ylabel("Test MAE")
    ax.set_title("Test MAE Mean ± Std Over 3 Random Seeds"); ax.legend()
    save_fig(fig, "fig5_seed_robustness")

# improvement map
if "imp" in globals():
    fig, ax = plt.subplots(figsize=(8, 8))
    imp.plot(column=imp["change_pct"].clip(-30, 30), cmap="RdBu_r", vmin=-30, vmax=30, ax=ax,
             legend=True, missing_kwds={"color": "lightgray"},
             legend_kwds={"label": "MAE change vs GRU (%)", "shrink": 0.6})
    ax.set_title("Where the Graph Helps (blue = GNN better)"); ax.axis("off")
    save_fig(fig, "fig6_improvement_map")

print("\nFigures saved to Drive:", sorted(os.listdir(FIG)))
print("\nPROJECT PIPELINE COMPLETE!")

In [ ]:
import requests, holidays, json
from zoneinfo import ZoneInfo

print(" DASHBOARD FORECAST ENGINE")
print("=" * 55)

NY = ZoneInfo("America/New_York")
slug = lambda n: re.sub(r"[^a-z0-9]+", "_", n.lower()).strip("_")
zone_list = final_demand_matrix.columns.tolist()
zone_info = zones.set_index("LocationID").loc[zone_list, ["Borough", "Zone"]]
zone_names = zone_info["Zone"].to_numpy()
zone_boro = zone_info["Borough"].to_numpy()
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# ---------- Best model  ----------
if "best_gnn" not in globals():
    _r = pd.read_csv(SAVE + "model_results.csv")
    _v = _r[(_r["split"] == "val") & (_r["subset"] == "all") &
            ~_r["model"].str.contains(r"\[", regex=True)].groupby("model")["MAE"].mean()
    best_gnn = _v[[m for m in _v.index if (m.startswith("GCN-GRU") and "random" not in m)
                   or m.startswith("Graph WaveNet")]].idxmin()
system_model = build_model(best_gnn).to(device)
system_model.load_state_dict(torch.load(SAVE + f"model_{slug(best_gnn)}.pt", map_location=device))
system_model.eval()
print("Forecasting model:", best_gnn)

# ---------- Weather scaling statistics  ----------
WX_CONT = ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m"]
w25 = pd.read_parquet(SAVE + "weather_2025.parquet")
wx_stats = {c: (w25.loc[w25.index < TRAIN_END, c].mean(),
                w25.loc[w25.index < TRAIN_END, c].std()) for c in WX_CONT}

# ---------- Recent "normal" demand pattern from the latest published TLC month ----------
def latest_tlc_month(max_back=18):
    now = pd.Timestamp.now(tz=NY).tz_localize(None).to_period("M")
    for k in range(max_back):
        p = now - k
        url = ("https://d37ci6vzurychx.cloudfront.net/trip-data/"
               f"yellow_tripdata_{p.year}-{p.month:02d}.parquet")
        try:
            if requests.head(url, timeout=15).status_code == 200:
                return p, url
        except requests.RequestException:
            pass
    raise RuntimeError("No recent TLC month found.")

def recent_typical_pattern():
    p, url = latest_tlc_month()
    cache = SAVE + f"recent_typical_{p}.npz"
    if os.path.exists(cache):
        z = np.load(cache)
        return p, z["typ"], float(z["daily"])
    print(f"Downloading latest published TLC month: {p} ...")
    t = pd.read_parquet(url, columns=["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID",
                                      "trip_distance", "fare_amount", "payment_type"])
    start, end = p.start_time, (p + 1).start_time
    base = ((t["tpep_pickup_datetime"] >= start) & (t["tpep_pickup_datetime"] < end) &
            t["PULocationID"].isin(zone_list))
    zero = t["tpep_dropoff_datetime"] <= t["tpep_pickup_datetime"]
    phantom = zero & ((t["payment_type"] == 6) | (t["fare_amount"] < 0) |
                      ((t["trip_distance"] <= 0) & (t["fare_amount"] <= 0)))   # same rule as Step 17
    t = t.loc[base & ~phantom]
    hourly = (t.groupby([t["tpep_pickup_datetime"].dt.floor("h"), "PULocationID"]).size()
               .unstack(fill_value=0)
               .reindex(index=pd.date_range(start, end, freq="h", inclusive="left"),
                        columns=zone_list, fill_value=0))
    last4 = hourly.iloc[-28 * 24:]                       # last 4 weeks of the month
    typ = (last4.groupby(last4.index.dayofweek * 24 + last4.index.hour).mean()
                .reindex(range(168)).fillna(0).to_numpy(np.float32))
    daily = hourly.to_numpy().sum() / (len(hourly) / 24)
    np.savez(cache, typ=typ, daily=daily)
    return p, typ, daily

TYP_MONTH, typ_pattern, recent_daily = recent_typical_pattern()
same_2025 = final_demand_matrix.loc[f"2025-{TYP_MONTH.month:02d}"]
level_change = recent_daily / (same_2025.to_numpy().sum() / (len(same_2025) / 24)) - 1
print(f"Recent pattern from {TYP_MONTH} | avg {recent_daily:,.0f} pickups/day "
      f"({level_change:+.1%} vs same month in 2025)")

# ---------- Weather for the forecast window (Open-Meteo forecast API) ----------
def get_weather(times):
    params = dict(latitude=40.7128, longitude=-74.006, hourly=",".join(WX_CONT),
                  timezone="America/New_York",
                  start_date=str(times.min().date()), end_date=str(times.max().date()))
    try:
        r = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=30)
        r.raise_for_status()
        w = pd.DataFrame(r.json()["hourly"])
        w["time"] = pd.to_datetime(w["time"])
        w = (w.drop_duplicates("time").set_index("time")
              .reindex(times).interpolate(limit_direction="both"))
        return w, "Open-Meteo hourly forecast"
    except Exception as e:
        w = pd.DataFrame({c: wx_stats[c][0] for c in WX_CONT}, index=times)
        return w, f"unavailable ({type(e).__name__}); training-period averages used"

# ---------- Context features ----------
def build_context(times, w):
    years = list(range(times.min().year - 1, times.max().year + 2))
    hol_days = pd.to_datetime(list(holidays.US(years=years).keys()))
    day = times.normalize()
    ctx = pd.DataFrame(index=times)
    ctx["hour_sin"] = np.sin(2 * np.pi * times.hour / 24)
    ctx["hour_cos"] = np.cos(2 * np.pi * times.hour / 24)
    ctx["dow_sin"] = np.sin(2 * np.pi * times.dayofweek / 7)
    ctx["dow_cos"] = np.cos(2 * np.pi * times.dayofweek / 7)
    ctx["is_weekend"] = (times.dayofweek >= 5).astype(int)
    ctx["is_holiday"] = day.isin(hol_days).astype(int)
    ctx["pre_holiday"] = (day + pd.Timedelta(days=1)).isin(hol_days).astype(int)
    ctx["post_holiday"] = (day - pd.Timedelta(days=1)).isin(hol_days).astype(int)
    ctx["dst_flag"] = 0
    for c in WX_CONT:
        ctx[c] = (w[c].to_numpy() - wx_stats[c][0]) / wx_stats[c][1]
    ctx["is_raining"] = (w["precipitation"].to_numpy() > 0.1).astype(int)
    return ctx[ctx_cols]

# ---------- 24-hour recursive forecast for a date ----------
@torch.no_grad()
def forecast_day(date):
    day0 = pd.Timestamp(date).normalize()
    times = pd.date_range(day0 - pd.Timedelta(hours=T_IN), periods=T_IN + 24, freq="h")
    w, wsrc = get_weather(times)
    ctx = build_context(times, w).to_numpy(np.float32)
    typical = typ_pattern[(times.dayofweek * 24 + times.hour).to_numpy()]      # [T_IN+24, N]

    window = list(((np.log1p(typical[:T_IN]) - mu) / sd).astype(np.float32))
    preds = []
    for k in range(24):
        xd = torch.tensor(np.stack(window[-T_IN:]), device=device)[None]       # [1, T_IN, N]
        xc = torch.tensor(ctx[k:k + T_IN], device=device)[None]                # [1, T_IN, F]
        nxt = system_model(xd, xc)[0, 0]                                       # next hour, scaled log
        preds.append(torch.expm1(nxt * sd_t + mu_t).clamp_min(0).cpu().numpy())
        window.append(nxt.cpu().numpy())

    hol = holidays.US(years=[day0.year]).get(day0.date())
    return {"date": day0, "times": times[T_IN:], "pred": np.stack(preds),      # [24, N]
            "typical": typical[T_IN:], "weather": w.iloc[T_IN:], "weather_source": wsrc,
            "holiday": hol}

def day_tables(res, min_typical=100):
    pt, tt = res["pred"].sum(0), res["typical"].sum(0)
    df = pd.DataFrame({"Zone": zone_names, "Borough": zone_boro, "LocationID": zone_list,
                       "forecast_day": pt.round(0), "typical_day": tt.round(0),
                       "change_%": np.where(tt > 0, (pt / np.maximum(tt, 1e-9) - 1) * 100, np.nan).round(1),
                       "peak_hour": [f"{h:02d}:00" for h in res["pred"].argmax(0)]})
    top = df.nlargest(10, "forecast_day")
    big = df[df["typical_day"] >= min_typical]
    return df, top, big.nlargest(10, "change_%"), big.nsmallest(10, "change_%")

# ---------- Test: forecast today ----------
today = pd.Timestamp.now(tz=NY).tz_localize(None).normalize()
res_today = forecast_day(today)
df_today, top_today, rise_today, fall_today = day_tables(res_today)
print(f"\nForecast for {today:%A %Y-%m-%d} | weather: {res_today['weather_source']} | "
      f"holiday: {res_today['holiday'] or 'none'}")
print(f"Citywide forecast: {res_today['pred'].sum():,.0f} pickups "
      f"(normal {DAYS[today.dayofweek]}: {res_today['typical'].sum():,.0f}, "
      f"{(res_today['pred'].sum() / res_today['typical'].sum() - 1) * 100:+.1f}%)")
display(top_today)

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(res_today["times"], res_today["typical"].sum(1), ls="--", color="gray", label="Normal pattern")
ax.plot(res_today["times"], res_today["pred"].sum(1), color="tab:red", marker="o", label="Model forecast")
ax.set_title(f"Citywide Hourly Forecast — {today:%a %b %d, %Y}"); ax.set_ylabel("Pickups/hour")
ax.legend(); plt.show()
print("\nFORECAST ENGINE READY!")

In [ ]:
!pip -q install sentence-transformers
import pandas as pd
import numpy as np
import os
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

print("RAG CHATBOT")
print("=" * 55)

res_all = pd.read_csv(SAVE + "model_results.csv")
main_res = res_all[~res_all["model"].str.contains(r"\[", regex=True)]
M = final_demand_matrix

# ---------- Static knowledge base ----------
def build_static_docs():
    docs = []
    add = lambda title, text, kind="project": docs.append({"title": title, "text": text, "kind": kind})

    add("Project overview",
        "This dashboard is a Spatiotemporal Urban Mobility Forecasting System for New York City. "
        "It predicts short-term yellow-taxi pickup demand for 263 NYC taxi zones using Graph Neural "
        "Networks (GNNs), which represent zones as nodes in a graph so the model can learn how demand "
        "in neighbouring or connected zones influences each other. It uses historical trips, time "
        "features, weather, holidays and zone geography, and compares GNNs with traditional "
        "machine-learning and time-series baselines.")
    add("Data used",
        f"The model was built from NYC TLC Yellow Taxi trip records for January–December 2025: "
        f"{int(counts.sum()):,} cleaned pickups aggregated into {T:,} hourly intervals × {N} taxi zones "
        "(five boroughs plus Newark Airport). Cleaning removed out-of-month timestamps, invalid zones and "
        "phantom zero-duration records (voided, negative-fare or zero-distance-zero-fare); fixed the "
        "daylight-saving hours; and imputed rare data gaps. Weather came from Open-Meteo.")
    add("Train, validation and test split",
        "Data was split chronologically: training January–September 2025, validation October 2025, "
        "test November–December 2025. Models forecast 1, 3 and 6 hours ahead from the previous 12 hours.")
    add("Features",
        "Inputs are the last 12 hours of log-scaled demand per zone, hour of day and day of week "
        "(sine/cosine encoded), weekend flag, federal holiday, day-before and day-after holiday flags, "
        "temperature, precipitation, snowfall, wind speed and a rain indicator.")
    add("Graphs",
        "Four zone graphs were tested: contiguity (zones sharing a border), distance (8 nearest zones "
        "with a Gaussian weight), OD flow (zones connected by many taxi trips, which links airports to "
        "Manhattan) and correlation (zones whose demand fluctuates together). A random graph was used as "
        "a control. Graph WaveNet can also learn its own adaptive graph from data.")
    add("Models compared",
        "Baselines: persistence, seasonal naive (yesterday, last week), historical average and 4-week "
        "moving average. Machine learning: LightGBM with and without neighbour (spatial lag) features. "
        "Deep learning: a GRU with no graph. Graph neural networks: GCN-GRU on each graph and Graph WaveNet "
        f"with an adaptive adjacency. The dashboard uses the best model on validation data: {best_gnn}.")
    add("Evaluation metrics",
        "MAE is the mean absolute error in pickups per zone per hour. RMSE penalises large errors more. "
        "WMAPE is total absolute error divided by total actual pickups (a percentage-style error that "
        "works even when many zone-hours have zero pickups). Lower is better for all three.")
    add("How the dashboard forecast works",
        f"Live taxi data is not public in real time: TLC publishes trip records monthly with a delay. "
        f"The dashboard therefore starts from the normal hourly pattern of the latest published month "
        f"({TYP_MONTH}), adds the real hourly weather forecast and holiday calendar for the chosen date, "
        f"and runs the GNN forward hour by hour for 24 hours. The 'new pattern' is how the forecast differs "
        f"from a normal day. Recent demand level is {level_change:+.1%} versus the same month in 2025.")
    add("Limitations",
        "Forecasts are model estimates. Only yellow taxis are included (no green taxis, Uber or Lyft). "
        "The model was trained on one year of data, holidays are learned from few examples, special events "
        "(concerts, games, parades) are not known to the model, and the dashboard cannot see today's actual "
        "trips, so sudden disruptions are not captured.")

    # Results per model
    for m in main_res["model"].unique():
        r = main_res[(main_res["model"] == m) & (main_res["split"] == "test")]
        a = r[r["subset"] == "all"].set_index("horizon")
        if a.empty:
            continue
        txt = (f"Test-set results (Nov–Dec 2025, all zones) for {m}: " +
               "; ".join(f"{h}-hour-ahead MAE {a.loc[h, 'MAE']:.3f}, RMSE {a.loc[h, 'RMSE']:.3f}, "
                         f"WMAPE {a.loc[h, 'WMAPE']:.1%}" for h in EVAL_H if h in a.index) + ".")
        add(f"Results: {m}", txt, "results")
    rank = (main_res[(main_res["split"] == "test") & (main_res["subset"] == "all") & (main_res["horizon"] == 1)]
            .sort_values("MAE")[["model", "MAE"]].head(6))
    add("Which model is best",
        f"The model selected on validation data is {best_gnn}. Ranking by 1-hour-ahead test MAE: " +
        "; ".join(f"{i + 1}. {r.model} ({r.MAE:.3f})" for i, r in enumerate(rank.itertuples())) + ".",
        "results")
    for fname, title in [("diebold_mariano_tests.csv", "Statistical significance (Diebold–Mariano)"),
                         ("seed_robustness.csv", "Random-seed robustness"),
                         ("context_ablation.csv", "Context-feature ablation")]:
        if os.path.exists(SAVE + fname):
            add(title, pd.read_csv(SAVE + fname).round(4).to_string(index=False), "results")

    # Borough and zone facts
    total = M.sum()
    ranks = total.rank(ascending=False).astype(int)
    wk = M.groupby(M.index.dayofweek * 24 + M.index.hour).mean()
    for b in ["Manhattan", "Queens", "Brooklyn", "Bronx", "Staten Island", "EWR"]:
        cols = [z for z in zone_list if zone_info.loc[z, "Borough"] == b]
        top_z = zone_info.loc[total[cols].idxmax(), "Zone"]
        add(f"Borough: {b}",
            f"{b} has {len(cols)} taxi zones and {int(total[cols].sum()):,} yellow-taxi pickups in 2025 "
            f"({total[cols].sum() / total.sum():.1%} of the city). Its busiest zone is {top_z}.", "borough")
    for j, z in enumerate(zone_list):
        s, peak = M[z], int(wk[z].idxmax())
        add(f"Zone: {zone_names[j]} ({zone_boro[j]})",
            f"{zone_names[j]} is taxi zone {z} in {zone_boro[j]}. In 2025 it had {int(s.sum()):,} pickups "
            f"(rank {ranks[z]} of {N}), averaging {s.mean():.1f} per hour; {(s == 0).mean():.0%} of hours had "
            f"no pickups. Its busiest time of week is {DAYS[peak // 24]} at {peak % 24:02d}:00 "
            f"(about {wk[z].max():.0f} pickups per hour). In {TYP_MONTH} it averaged about "
            f"{typ_pattern[:, j].sum() / 7:.0f} pickups per day.", "zone")
    return docs

STATIC_DOCS = build_static_docs()
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(device))
embed = lambda docs: embedder.encode([d["title"] + ". " + d["text"] for d in docs],
                                     normalize_embeddings=True, show_progress_bar=False)
STATIC_EMB = embed(STATIC_DOCS)
DYN_DOCS, DYN_EMB = [], np.zeros((0, STATIC_EMB.shape[1]), dtype=np.float32)
print(f"Static knowledge base: {len(STATIC_DOCS)} documents")

# ---------- Dynamic documents: the latest dashboard forecast ----------
def update_dynamic_docs(res):
    global DYN_DOCS, DYN_EMB
    d = res["date"]; pred, typ = res["pred"], res["typical"]
    df, top, rise, fall = day_tables(res)
    w = res["weather"]
    above = "; ".join(f"{z} {c:+.1f}%" for z, c in zip(rise["Zone"], rise["change_%"]))
    below = "; ".join(f"{z} {c:+.1f}%" for z, c in zip(fall["Zone"], fall["change_%"]))
    busiest = "; ".join(f"{z} ({b}) {v:,.0f}"
                        for z, b, v in zip(top["Zone"], top["Borough"], top["forecast_day"]))
    docs = [{"title": f"Citywide forecast for {d:%A %B %d, %Y}", "kind": "forecast",
             "text": f"For {d:%A %B %d, %Y} the model forecasts {pred.sum():,.0f} yellow-taxi pickups citywide, "
                     f"versus {typ.sum():,.0f} on a normal {DAYS[d.dayofweek]} "
                     f"({(pred.sum() / typ.sum() - 1) * 100:+.1f}%). Peak hour: {int(pred.sum(1).argmax()):02d}:00 "
                     f"with {pred.sum(1).max():,.0f} pickups. Weather: {w['temperature_2m'].min():.0f}–"
                     f"{w['temperature_2m'].max():.0f} °C, {w['precipitation'].sum():.1f} mm rain, "
                     f"{w['snowfall'].sum():.1f} cm snow. Holiday: {res['holiday'] or 'none'}."},
            {"title": f"Busiest zones forecast for {d:%B %d, %Y}", "kind": "forecast",
             "text": f"Highest forecast pickups: {busiest}."},
            {"title": f"Zones above and below normal on {d:%B %d, %Y}", "kind": "forecast",
             "text": f"Most above normal: {above}. Most below normal: {below}."}]
    for j in range(N):
        hrs = ", ".join(f"{h:02d}:00 {pred[h, j]:.0f}" for h in range(0, 24, 2))
        docs.append({"title": f"Forecast for {zone_names[j]} on {d:%B %d, %Y}", "kind": "forecast",
                     "text": f"{zone_names[j]} ({zone_boro[j]}) is forecast to have {pred[:, j].sum():.0f} pickups "
                             f"on {d:%A %B %d, %Y} (normal: {typ[:, j].sum():.0f}). Peak at "
                             f"{int(pred[:, j].argmax()):02d}:00 with {pred[:, j].max():.0f} pickups. "
                             f"Every two hours: {hrs}."})
    DYN_DOCS, DYN_EMB = docs, embed(docs)

update_dynamic_docs(res_today)

def retrieve(q, k=6):
    docs = STATIC_DOCS + DYN_DOCS
    emb = np.vstack([STATIC_EMB, DYN_EMB])
    scores = emb @ embedder.encode([q], normalize_embeddings=True)[0]
    return [docs[i] for i in np.argsort(-scores)[:k]]

# ---------- Local LLM for answer generation ----------
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
try:
    tok = AutoTokenizer.from_pretrained(LLM_NAME)
    llm = AutoModelForCausalLM.from_pretrained(
        LLM_NAME, torch_dtype=torch.float16 if device.type == "cuda" else torch.float32).to(device).eval()
    print("LLM loaded:", LLM_NAME)
except Exception as e:
    llm = None
    print(f"LLM unavailable ({type(e).__name__}); the chatbot will return the best-matching passage instead.")

SYSTEM = ("You are the assistant for an NYC taxi demand forecasting dashboard built for a data science "
          "capstone project. Answer ONLY from the numbered context passages. Copy numbers exactly. "
          "If the context does not contain the answer, say you don't have that information. "
          "Answer in 2–5 short sentences. Forecasts are model estimates, not guarantees.")

@torch.no_grad()
def answer(question, history=None):
    hits = retrieve(question)
    sources = "\n\n*Sources: " + "; ".join(d["title"] for d in hits[:3]) + "*"
    if llm is None:
        return hits[0]["text"] + sources

    context = "\n\n".join(f"[{i + 1}] {d['title']}\n{d['text']}" for i, d in enumerate(hits))
    msgs = [{"role": "system", "content": SYSTEM}]
    for turn in (history or [])[-4:]:                      # short memory for follow-up questions
        if isinstance(turn, dict) and turn.get("role") in ("user", "assistant"):
            msgs.append({"role": turn["role"], "content": str(turn["content"])[:500]})
    msgs.append({"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"})

    # Format the chat as text, then tokenize (works across transformers versions)
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = tok(prompt, return_tensors="pt").to(device)
    out = llm.generate(**enc, max_new_tokens=300, do_sample=False,
                       pad_token_id=tok.eos_token_id)
    reply = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return reply + sources

for q in ["How many pickups are forecast citywide today?",
          "What is the busiest zone in Queens?",
          "Which model performed best and what is WMAPE?",
          "Why can't the dashboard use live taxi data?"]:
    print(f"\nQ: {q}\nA: {answer(q)}")
print("\nRAG CHATBOT READY!")

In [ ]:
!pip -q install "gradio>=5,<6"
import gradio as gr, plotly.express as px, plotly.graph_objects as go

print("STEP 31: INTERACTIVE DASHBOARD")
print("=" * 55)

# ---------- Map geometry (simplified for fast rendering) ----------
geo = gdf[["geometry"]].copy()
geo["geometry"] = geo.geometry.simplify(150)            # 150 ft tolerance (EPSG:2263)
geo = geo.reset_index(drop=True).to_crs(4326)
geo["LocationID"] = [int(z) for z in zone_list]
GEOJSON = json.loads(geo.to_json())

def choropleth(df, **kw):
    common = dict(geojson=GEOJSON, locations="LocationID", featureidkey="properties.LocationID",
                  center={"lat": 40.73, "lon": -73.94}, zoom=9.3, opacity=0.75, height=560)
    if hasattr(px, "choropleth_map"):
        return px.choropleth_map(df, map_style="carto-positron", **common, **kw)
    return px.choropleth_mapbox(df, mapbox_style="carto-positron", **common, **kw)

# ---------- Figures ----------
def city_fig(res):
    t = res["times"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t, y=res["typical"].sum(1), name="Normal day", line=dict(dash="dash", color="gray")))
    fig.add_trace(go.Scatter(x=t, y=res["pred"].sum(1), name="Forecast", mode="lines+markers", line=dict(color="crimson")))
    fig.update_layout(title="Citywide pickups per hour", height=360, yaxis_title="Pickups",
                      margin=dict(l=40, r=10, t=50, b=30), legend=dict(orientation="h"))
    return fig

def zone_fig(res, zone):
    j = zone_list.index(int(zone))
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=res["times"], y=res["typical"][:, j], name="Normal day", line=dict(dash="dash", color="gray")))
    fig.add_trace(go.Scatter(x=res["times"], y=res["pred"][:, j], name="Forecast", mode="lines+markers", line=dict(color="royalblue")))
    fig.update_layout(title=f"{zone_names[j]} ({zone_boro[j]})", height=360, yaxis_title="Pickups",
                      margin=dict(l=40, r=10, t=50, b=30), legend=dict(orientation="h"))
    return fig

def forecast_map(res):
    long = pd.DataFrame({"LocationID": np.tile([int(z) for z in zone_list], 24),
                         "Zone": np.tile(zone_names, 24), "Borough": np.tile(zone_boro, 24),
                         "Hour": np.repeat([t.strftime("%H:00") for t in res["times"]], N),
                         "Forecast": res["pred"].ravel().round(1),
                         "Normal": res["typical"].ravel().round(1)})
    fig = choropleth(long, color="Forecast", animation_frame="Hour", hover_name="Zone",
                     hover_data={"Borough": True, "Forecast": True, "Normal": True, "LocationID": False, "Hour": False},
                     color_continuous_scale="YlOrRd", range_color=(0, float(np.quantile(res["pred"], 0.99))))
    fig.update_layout(title="Forecast pickups by zone — press ▶ to play through the day",
                      margin=dict(l=0, r=0, t=50, b=0))
    return fig

def deviation_map(df):
    d = df.copy()
    d.loc[d["typical_day"] < 50, "change_%"] = np.nan        # hide very quiet zones
    fig = choropleth(d, color="change_%", hover_name="Zone",
                     hover_data={"Borough": True, "forecast_day": True, "typical_day": True, "LocationID": False},
                     color_continuous_scale="RdBu_r", range_color=(-30, 30))
    fig.update_layout(title="New pattern: forecast vs a normal day (%) — red = busier than normal",
                      margin=dict(l=0, r=0, t=50, b=0))
    return fig

# ---------- Dashboard callback ----------
LAST = {"res": res_today}
def run_dashboard(date_str, zone):
    today = pd.Timestamp.now(tz=NY).tz_localize(None).normalize()
    try:
        d = pd.Timestamp(date_str).normalize()
    except Exception:
        return ("Please enter a date as YYYY-MM-DD.",) + (None,) * 7
    if not (today - pd.Timedelta(days=60) <= d <= today + pd.Timedelta(days=14)):
        return (f"Choose a date between {today - pd.Timedelta(days=60):%Y-%m-%d} and "
                f"{today + pd.Timedelta(days=14):%Y-%m-%d} (weather forecasts reach 14 days ahead).",) + (None,) * 7

    res = forecast_day(d); LAST["res"] = res
    update_dynamic_docs(res)                                   # chatbot now knows this forecast
    df, top, rise, fall = day_tables(res)
    w = res["weather"]
    tot, typ = res["pred"].sum(), res["typical"].sum()
    summary = (
        f"### Forecast for {d:%A, %B %d, %Y}\n"
        f"- **Citywide pickups:** {tot:,.0f} (normal {DAYS[d.dayofweek]}: {typ:,.0f}, **{(tot / typ - 1) * 100:+.1f}%**)\n"
        f"- **Peak hour:** {int(res['pred'].sum(1).argmax()):02d}:00 with {res['pred'].sum(1).max():,.0f} pickups\n"
        f"- **Weather:** {w['temperature_2m'].min():.0f}–{w['temperature_2m'].max():.0f} °C · "
        f"rain {w['precipitation'].sum():.1f} mm · snow {w['snowfall'].sum():.1f} cm *({res['weather_source']})*\n"
        f"- **Holiday:** {res['holiday'] or 'none'}\n"
        f"- **Normal pattern from:** last 4 weeks of {TYP_MONTH} TLC data (latest published; "
        f"{level_change:+.1%} vs same month 2025)\n\n"
        f"*Model: {best_gnn}. Live trip data is not public, so forecasts combine the recent normal pattern "
        f"with this date's weather and calendar.*")
    cols = ["Zone", "Borough", "forecast_day", "typical_day", "change_%", "peak_hour"]
    return (summary, city_fig(res), zone_fig(res, zone), forecast_map(res), deviation_map(df),
            top[cols], rise[cols], fall[cols])

def chat_fn(message, history):
    return answer(message, history)

# ---------- About tab ----------
key = [m for m in main_res["model"].unique()
       if m in (best_gnn, "GRU (no graph)", "LightGBM + spatial lags", "LightGBM (no spatial features)")
       or m.startswith(("Weekly moving", "Historical"))]
about_tbl = (main_res[(main_res["split"] == "test") & (main_res["subset"] == "all") & main_res["model"].isin(key)]
             .pivot_table(index="model", columns="horizon", values="MAE").round(3))
about_tbl.columns = [f"MAE {h}h" for h in about_tbl.columns]
about_md = (
    "## About the model\n"
    "This system forecasts hourly yellow-taxi pickups for all 263 NYC taxi zones with a Graph Neural Network "
    "that treats zones as connected nodes, so demand in one area informs forecasts for related areas.\n\n"
    f"**Deployed model:** {best_gnn} (selected on October 2025 validation data)\n\n"
    "**Training data:** NYC TLC Yellow Taxi trips, Jan–Sep 2025 · validation Oct 2025 · test Nov–Dec 2025\n\n"
    "### Test-set accuracy (mean absolute error, pickups per zone-hour)\n" + about_tbl.to_markdown() +
    "\n\n### Limitations\n- Yellow taxis only (no green taxis or ride-hail)\n- Live trip data is not public; "
    "forecasts use the latest published month as the normal pattern\n- Special events are not known to the model\n"
    "- Later hours of the 24-hour forecast are less certain because predictions feed into the next step")

zone_choices = sorted([(f"{zone_names[j]} ({zone_boro[j]})", int(z)) for j, z in enumerate(zone_list)])
today_str = pd.Timestamp.now(tz=NY).strftime("%Y-%m-%d")

with gr.Blocks(title="NYC Taxi Demand Forecasting", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# NYC Taxi Demand Forecasting Dashboard\n"
                "Spatiotemporal forecasting with Graph Neural Networks · CS668 Analytics Capstone, Pace University")
    with gr.Tab("Forecast"):
        with gr.Row():
            date_in = gr.Textbox(value=today_str, label="Date (YYYY-MM-DD)", scale=1)
            zone_in = gr.Dropdown(choices=zone_choices, value=161, label="Zone to inspect", scale=2)
            run_btn = gr.Button("Run forecast", variant="primary", scale=1)
        summary_out = gr.Markdown()
        with gr.Row():
            city_out = gr.Plot()
            zone_out = gr.Plot()
        with gr.Row():
            map_out = gr.Plot()
            dev_out = gr.Plot()
        gr.Markdown("### Busiest zones · biggest increases · biggest decreases (vs a normal day)")
        with gr.Row():
            top_out = gr.Dataframe(wrap=True)
            rise_out = gr.Dataframe(wrap=True)
            fall_out = gr.Dataframe(wrap=True)
    with gr.Tab("Ask the assistant"):
        gr.ChatInterface(
            fn=chat_fn, type="messages",
            examples=["How many pickups are forecast citywide today?",
                      "Which zones are busier than normal today?",
                      "What is the forecast for JFK Airport?",
                      "Which model does the dashboard use and how accurate is it?",
                      "What does WMAPE mean?"])
    with gr.Tab("About the model"):
        gr.Markdown(about_md)

    outputs = [summary_out, city_out, zone_out, map_out, dev_out, top_out, rise_out, fall_out]
    run_btn.click(run_dashboard, [date_in, zone_in], outputs)
    zone_in.change(lambda z: zone_fig(LAST["res"], z), zone_in, zone_out)
    demo.load(run_dashboard, [date_in, zone_in], outputs)       # forecast today on open

demo.launch(share=True, debug=False)